# 05 — Cross-study transfer (Smoke Free, EMASENS, alcohol)

Next-entry prediction on 17 features common to the three studies. Parts used in the paper:

| Part | Cells | Paper |
|---|---|---|
| 1–4: transfer between studies, fold-matched gaps | 1–12 | Supplement S2a–S2b |
| 8–9: pooled training, partial pooling by feature augmentation | 18–23 | Supplement S2c |
| 9D: small target studies (3–40 participants; 5 folds × 3 repeats) | 24–25, then the Table 10 cell | Table 10, Figure 6, Supplement S2d |

Parts 5–7, 10 and 11 are exploratory or belong to the companion within-person paper and are not reported in this paper.
Set `SAVE_DIR` in Cell 1 (results are written there and finished runs are skipped after a restart).

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ════════════════════════════════════════════════════════
import os, json, time, random, subprocess, sys
import numpy as np, pandas as pd
try:
    import xgboost as xgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'xgboost', '--quiet']); import xgboost as xgb
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

SAVE_DIR = '/content/drive/MyDrive/transfer_results'   # results on Google Drive survive a disconnect
DATA_DIRS = ['.', '../data', '/content', '/content/drive/MyDrive', '/content/drive/MyDrive/transfer_data']

# Mount Drive automatically in Colab (does nothing elsewhere)
if SAVE_DIR.startswith('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive; drive.mount('/content/drive')
    except ImportError:
        SAVE_DIR = 'transfer_results'      # not in Colab: save locally


def find_file(fname):
    """Looks for a data file in /content (uploaded) and then on Drive, so a reconnect only needs Run all."""
    for d in DATA_DIRS:
        pth = os.path.join(d, fname)
        if os.path.exists(pth): return pth
    raise FileNotFoundError(f"{fname} not found in {DATA_DIRS}. Upload it, or copy it to Drive "
                            f"(MyDrive/transfer_data/) so it survives disconnects.")


SF_PATH  = find_file('smokefree_data.xlsx')
EM_PATH  = find_file('analytic_sample_1.csv')
AL_PATH  = find_file('ema.csv')
AL_IDS   = find_file('baseline_features_1week_v3.csv')

SEED       = 42
QUICK_TEST = False      # True: 60 SF / 12 EM / 20 AL participants, 1 epoch, 1 repeat
BIN_MODE   = 'cutoff'   # 'cutoff' | 'prevalence'
BIN_CUT    = 6
EPOCHS     = 1 if QUICK_TEST else 30
FT_EPOCHS  = 1 if QUICK_TEST else 15     # fine-tuning epochs (E3)
N_FOLDS    = 2 if QUICK_TEST else 10     # within-dataset CV (E1, E2)
E3_FOLDS   = 2 if QUICK_TEST else 5
E3_REPEATS = 1 if QUICK_TEST else 3
E3_N = {'EM': [5, 10] if QUICK_TEST else [5, 10, 20, 30],
        'AL': [5, 10] if QUICK_TEST else [10, 20, 40, 80, 120]}
N_BOOT     = 200 if QUICK_TEST else 1000
if QUICK_TEST: SAVE_DIR += '_quicktest'
os.makedirs(os.path.join(SAVE_DIR, 'predictions'), exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device={device}  QUICK_TEST={QUICK_TEST}  BIN_MODE={BIN_MODE}  results -> {SAVE_DIR}")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — LOAD AND HARMONISE THE THREE DATASETS
# Each dataset becomes one table: id, time (local), label + harmonised columns.
# ════════════════════════════════════════════════════════
EPOCH = pd.Timestamp('1970-01-01')
HARM_ITEMS = ['craving', 'anxious', 'bored', 'happy', 'annoyed', 'down',
              'act_working', 'act_reading', 'act_relaxing', 'act_eating', 'act_social',
              'caffeine', 'alcohol', 'driving', 'soc_alone', 'soc_partner', 'soc_friends', 'soc_family',
              'cpd', 'ttfc', 'days_since_start', 'mins_since_prev', 'prior_lapse']          # 23
FEELINGS = ['anxious', 'bored', 'happy', 'annoyed', 'down']
EXTRA = ['hour_sin', 'hour_cos', 'is_weekend', 'severity_change', 'rolling3_sev_mean',
         'rolling3_sev_std', 'days_since_last_lapse', 'log_days_since_qd',
         'sev_lag1', 'sev_lag2', 'sev_lag3', 'lapse_lag2', 'lapse_lag3']                     # 13
CORE = ['craving', 'prior_lapse', 'mins_since_prev', 'days_since_start'] + EXTRA             # 17


def add_common(d):
    """Sort, then add the lapse-history, timing and 13 engineered features (same definitions everywhere)."""
    d = d.sort_values(['id', 't']).reset_index(drop=True)
    g = d.groupby('id', sort=False)
    d['prior_lapse'] = g['label'].shift(1).fillna(0)
    d['mins_since_prev'] = g['t'].diff().dt.total_seconds().div(60).fillna(0).clip(lower=0)
    hour = d['t'].dt.hour.astype(float)
    d['hour_sin'] = np.sin(2 * np.pi * hour / 24); d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['is_weekend'] = d['t'].dt.dayofweek.ge(5).astype(float)
    d['severity_change'] = g['craving'].diff().fillna(0)
    d['rolling3_sev_mean'] = g['craving'].transform(lambda v: v.shift(1).rolling(3, min_periods=1).mean()).fillna(0)
    d['rolling3_sev_std'] = g['craving'].transform(lambda v: v.shift(1).rolling(3, min_periods=1).std()).fillna(0)
    ts = (d['t'] - EPOCH).dt.total_seconds()
    start = (g['t'].transform('min') - EPOCH).dt.total_seconds()
    lt = pd.Series(np.where(d['label'].eq(1), ts, np.nan), index=d.index)
    last = lt.groupby(d['id'], sort=False).transform(lambda v: v.shift(1).ffill())
    d['days_since_last_lapse'] = ((ts - last.fillna(start)) / 86400).clip(lower=0)
    d['log_days_since_qd'] = np.log1p(d['days_since_start'].clip(lower=0))
    for k in (1, 2, 3): d[f'sev_lag{k}'] = g['craving'].shift(k).fillna(0)
    for k in (2, 3):    d[f'lapse_lag{k}'] = g['label'].shift(k).fillna(0)
    # person-level baseline: running lapse rate up to and including this entry (used to score entry t+1)
    d['run_rate'] = g['label'].transform(lambda v: v.expanding().mean())
    return d


def load_sf(path):
    r = pd.read_excel(path, sheet_name='All_Data')
    n = r.groupby('account_id').size(); r = r[r['account_id'].isin(n[n >= 20].index)]
    m = lambda c: r[c].astype(float)
    d = pd.DataFrame({'id': 'SF_' + r['account_id'].astype(str),
                      't': pd.to_datetime(r['adjusted_craving_record_created']),
                      'label': r['craving_did_smoke'].astype(str).str.lower().map({'true': 1., 'false': 0.}),
                      'craving': m('craving_severity'),
                      'anxious': m('craving_feeling_anxious'), 'bored': m('craving_feeling_bored'),
                      'happy': m('craving_feeling_happy'), 'annoyed': m('craving_feeling_annoyed'),
                      'down': m('craving_feeling_down'),
                      'act_working': m('craving_occupation_working'), 'act_reading': m('craving_occupation_reading'),
                      'act_relaxing': m('craving_occupation_relaxing'), 'act_eating': m('craving_occupation_just_eaten'),
                      'act_social': m('craving_occupation_chatting'), 'caffeine': m('craving_occupation_coffee_or_tea'),
                      'alcohol': m('craving_occupation_drinking'), 'driving': m('craving_occupation_driving'),
                      'soc_alone': m('craving_env_alone'), 'soc_partner': m('craving_env_partner'),
                      'soc_friends': m('craving_env_friends'), 'soc_family': m('craving_env_family'),
                      'cpd': m('cigs_per_day'),
                      'ttfc': r['time_to_first_cig'].map({'<= 5': 0, '6-30': 1, '31-60': 2, '> 60': 3}).astype(float),
                      'days_since_start': m('days_since_qd')})
    for k_, c_ in SF_EXTRA_COLS.items(): d[k_] = m(c_).to_numpy()       # extra items (Part 8 only)
    d['stressed'] = m('craving_feeling_stressed').to_numpy()
    return add_common(d)


SF_EXTRA_COLS = {'sf_hungry': 'craving_feeling_hungry', 'sf_lonely': 'craving_feeling_lonely',
                 'sf_bar': 'craving_occupation_at_bar_or_event', 'sf_bed': 'craving_occupation_going_to_bed',
                 'sf_sex': 'craving_occupation_just_had_sex', 'sf_break': 'craving_occupation_taking_a_break',
                 'sf_thinking': 'craving_occupation_thinking', 'sf_waking': 'craving_occupation_waking_up',
                 'sf_colleagues': 'craving_env_colleagues', 'sf_strangers': 'craving_env_strangers'}
EM_EXTRA_COLS = {'em_excited': 'excited_imp', 'em_calm': 'calm_imp', 'em_enthusiastic': 'enthusiastic_imp',
                 'em_contented': 'contented_imp', 'em_motivation': 'motivation_imp', 'em_confidence': 'confidence_imp',
                 'em_pain': 'pain_imp', 'em_cig_avail': 'cigarette_availability_imp', 'em_nicotine': 'nicotine_imp'}
AL_EXTRA_COLS = {'al_risky': 'ema_3', 'al_stressful': 'ema_4', 'al_pleasant': 'ema_5', 'al_valence': 'ema_6',
                 'al_arousal': 'ema_7'}


def load_em(path, sf_rates=None):
    r = pd.read_csv(path, parse_dates=['EMA_date_time'], dtype={'id': str})
    r = r[r['EMA_date_time'].notna()]                        # answered prompts only
    m = lambda c: r[c].astype(float)
    d = pd.DataFrame({'id': 'EM_' + r['id'], 't': r['EMA_date_time'], 'label': m('lapse_event_imp'),
                      'craving': m('craving_imp'),
                      'anxious': m('anxious_imp'), 'bored': m('bored_imp'), 'happy': m('happy_imp'),
                      'annoyed': m('irritable_imp'), 'down': m('sad_imp'),
                      'act_working': m('activity_working_imp'), 'act_reading': m('activity_reading_imp'),
                      'act_relaxing': m('activity_relaxing_imp'), 'act_eating': m('activity_eating_imp'),
                      'act_social': m('activity_socialising_imp'), 'caffeine': m('caffeine_imp'),
                      'alcohol': m('alcohol_imp'), 'driving': m('location_private_vehicle_imp'),
                      'soc_alone': m('social_context_alone_imp'), 'soc_partner': m('social_context_partner_imp'),
                      'soc_friends': m('social_context_friend_imp'), 'soc_family': m('social_context_relative_imp'),
                      'cpd': m('cpd'),
                      'ttfc': r['time_to_first_cig'].map({'Within 5 minutes': 0, '6-30 minutes': 1,
                                                          '31-60 minutes': 2, 'After 60 minutes': 3}).astype(float),
                      'days_since_start': m('study_day')})
    for k_, c_ in EM_EXTRA_COLS.items():                       # extra items (Part 8 only)
        v_ = pd.to_numeric(r[c_], errors='coerce')
        if v_.isna().all(): v_ = pd.Series(pd.factorize(r[c_])[0], index=r.index).astype(float)
        d[k_] = v_.fillna(v_.median()).to_numpy()
    d['stressed'] = (m('stressed_imp') >= BIN_CUT).astype(float).to_numpy()
    for f in FEELINGS:                                       # 0-10 ratings -> yes/no, as in SF
        if BIN_MODE == 'prevalence' and sf_rates is not None:
            cut = d[f].quantile(1 - sf_rates[f])
            d[f] = (d[f] >= cut).astype(float)
        else:
            d[f] = (d[f] >= BIN_CUT).astype(float)
    return add_common(d)


def load_al(path, ids_path):
    r = pd.read_csv(path, low_memory=False)
    ids = pd.read_csv(ids_path, usecols=['subid', 'dttm_label'])
    ids['dttm_label'] = pd.to_datetime(ids['dttm_label'], utc=True)
    per = ids.groupby('subid')['dttm_label'].agg(['min', 'max'])
    r['tu'] = pd.to_datetime(r['dttm_obs'], utc=True)
    r = r[r['subid'].isin(per.index)].join(per, on='subid')
    r = r[(r['tu'] >= r['min'] - pd.Timedelta(days=1)) & (r['tu'] <= r['max'] + pd.Timedelta(days=7))]
    t = r['tu'].dt.tz_convert('America/Chicago').dt.tz_localize(None)
    d = pd.DataFrame({'id': 'AL_' + r['subid'].astype(str), 't': t,
                      'label': (r['ema_1'] == 'Yes').astype(float),
                      'craving': r['ema_2'].astype(float) * 10 / 12})      # 0-12 -> 0-10
    for k_, c_ in AL_EXTRA_COLS.items(): d[k_] = pd.to_numeric(r[c_], errors='coerce').to_numpy()
    d = d.sort_values(['id', 't'])
    for k_ in AL_EXTRA_COLS:
        d[k_] = d.groupby('id')[k_].ffill(); d[k_] = d[k_].fillna(d[k_].median())
    d['craving'] = d.groupby('id')['craving'].ffill().fillna(d['craving'].median())
    d['days_since_start'] = (d['t'] - d.groupby('id')['t'].transform('min')).dt.total_seconds() / 86400
    return add_common(d)


def subsample(d, k):
    keep = sorted(d['id'].unique())[:k]
    return d[d['id'].isin(keep)].reset_index(drop=True)


DATA = {'SF': load_sf(SF_PATH)}
sf_rates = {f: DATA['SF'][f].mean() for f in FEELINGS}
DATA['EM'] = load_em(EM_PATH, sf_rates)
DATA['AL'] = load_al(AL_PATH, AL_IDS)
if QUICK_TEST:
    DATA = {'SF': subsample(DATA['SF'], 60), 'EM': subsample(DATA['EM'], 12), 'AL': subsample(DATA['AL'], 20)}

FSETS = {'harm': HARM_ITEMS, 'harm_exp': HARM_ITEMS + EXTRA, 'core': CORE}
for name, d in DATA.items():
    need = HARM_ITEMS + EXTRA if name != 'AL' else CORE
    assert not d[need].isna().any().any(), f"{name}: missing values in {d[need].columns[d[need].isna().any()].tolist()}"
    n = d.groupby('id').size(); lap = d.groupby('id')['label'].sum()
    gap = d.groupby('id')['t'].diff().dt.total_seconds().div(60).dropna()
    print(f"{name}: {d['id'].nunique()} people, {len(d):,} entries, lapse rate {d['label'].mean():.2%}, "
          f"people with >=1 lapse {(lap > 0).sum()}, gap median {gap.median():.0f} min")
print("\nFeeling prevalence, SF vs EM after conversion:")
print(pd.DataFrame({'SF': [DATA['SF'][f].mean() for f in FEELINGS],
                    'EM': [DATA['EM'][f].mean() for f in FEELINGS]}, index=FEELINGS).round(3).to_string())


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — MODELS (main-study settings) AND SEQUENCES
# ════════════════════════════════════════════════════════
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)


def seqs_of(d, feats, ids=None):
    """[(X, y, id, run_rate)] per person; next-entry pairs are built at train/predict time."""
    if ids is not None: d = d[d['id'].isin(ids)]
    return [(g[feats].to_numpy(np.float32), g['label'].to_numpy(np.float32), pid, g['run_rate'].to_numpy())
            for pid, g in d.groupby('id', sort=True) if len(g) >= 2]


def stats_of(seqs):
    X = np.vstack([s[0] for s in seqs]); return X.mean(0), X.std(0) + 1e-6


def standardise(seqs, mu, sd):
    return [((s[0] - mu) / sd, s[1], s[2], s[3]) for s in seqs]


def flat_next(seqs):
    X = np.vstack([s[0][:-1] for s in seqs]); y = np.concatenate([s[1][1:] for s in seqs])
    return X, y


def fit_flat(mtype, seqs):
    X, y = flat_next(seqs); y = y.astype(int)
    pos = max((y == 1).sum(), 1); neg = (y == 0).sum()
    if mtype == 'LR':
        clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
    elif mtype == 'RF':
        clf = RandomForestClassifier(n_estimators=500, min_samples_leaf=5, class_weight='balanced',
                                     n_jobs=-1, random_state=42)
    else:
        clf = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, subsample=0.8,
                                colsample_bytree=0.8, scale_pos_weight=neg / pos, eval_metric='logloss',
                                n_jobs=-1, random_state=42, verbosity=0)
    clf.fit(X, y)
    return clf


def predict_flat(clf, seqs):
    return [(clf.predict_proba(s[0][:-1])[:, 1], s[1][1:], s[2], s[3][:-1]) for s in seqs]


class GRUNet(nn.Module):
    def __init__(self, nf, hid=64, drop=0.3):
        super().__init__()
        self.gru = nn.GRU(nf, hid, 2, batch_first=True, dropout=drop)
        self.head = nn.Sequential(nn.Linear(hid, hid // 2), nn.GELU(), nn.Dropout(drop), nn.Linear(hid // 2, 1))
    def forward(self, x, lengths=None):
        if lengths is None: h = self.gru(x)[0]
        else:
            pk = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            h = pad_packed_sequence(self.gru(pk)[0], batch_first=True, total_length=x.size(1))[0]
        return self.head(h).squeeze(-1)


def collate(batch):
    xs, ys = zip(*batch); L = torch.tensor([len(x) for x in xs]); T = int(L.max())
    X = torch.zeros(len(xs), T, xs[0].shape[1]); Y = torch.full((len(xs), T), float('nan'))
    for i, (x, y) in enumerate(zip(xs, ys)):
        X[i, :len(x)] = torch.from_numpy(x); Y[i, :len(x)] = torch.from_numpy(y)
    return X, Y, L


def train_gru(seqs, nf, epochs, lr=1e-3, init=None, bs=32):
    net = GRUNet(nf).to(device)
    if init is not None: net.load_state_dict(init)
    y = np.concatenate([s[1][1:] for s in seqs])
    pw = torch.tensor([(y == 0).sum() / max((y == 1).sum(), 1)]).clamp(max=20).to(device)
    lf = nn.BCEWithLogitsLoss(pos_weight=pw); opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-4)
    dl = DataLoader([(s[0], s[1]) for s in seqs], batch_size=min(bs, len(seqs)), shuffle=True, collate_fn=collate)
    for _ in range(epochs):
        net.train()
        for X, Y, L in dl:
            X, Y = X.to(device), Y.to(device)
            pr, tg = net(X, L)[:, :-1], Y[:, 1:]; mk = ~torch.isnan(tg)
            if mk.sum() == 0: continue
            loss = lf(pr[mk], tg[mk]); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
    return net


def predict_gru(net, seqs):
    net.eval(); out = []
    with torch.no_grad():
        for X, y, pid, rr in seqs:
            p = torch.sigmoid(net(torch.from_numpy(X).unsqueeze(0).to(device))).squeeze(0).cpu().numpy()
            out.append((np.atleast_1d(p)[:-1], y[1:], pid, rr[:-1]))
    return out


def to_rows(preds, **tags):
    rows = []
    for p, y, pid, rr in preds:
        rows += [dict(tags, id=pid, y=int(a), p=float(b), base=float(c)) for a, b, c in zip(y, p, rr)]
    return rows
print("✅ models defined")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — EVALUATION HELPERS (pooled AUC, per-participant AUC, person baseline, bootstrap)
# ════════════════════════════════════════════════════════
def auc(y, p):
    return roc_auc_score(y, p) if 0 < np.sum(y) < len(y) else np.nan


def per_person(d, col='p'):
    return pd.Series({pid: auc(g['y'], g[col]) for pid, g in d.groupby('id')}).dropna()


def boot_pooled(d, cols=('p',), n_boot=N_BOOT, seed=SEED):
    """Participant-level bootstrap of pooled AUC; with two cols returns the CI of their difference."""
    rng = np.random.default_rng(seed); ids = d['id'].unique(); parts = {i: g for i, g in d.groupby('id')}
    vals = []
    for _ in range(n_boot):
        s = pd.concat([parts[i] for i in rng.choice(ids, len(ids))])
        v = [auc(s['y'], s[c]) for c in cols]
        vals.append(v[0] if len(v) == 1 else v[0] - v[1])
    return np.nanpercentile(vals, [2.5, 97.5])


def summarise_preds(d, label):
    pp, pb = per_person(d), per_person(d, 'base')
    lo, hi = boot_pooled(d)
    return {'config': label, 'n_people': d['id'].nunique(), 'pooled_AUC': round(auc(d['y'], d['p']), 3),
            'CI_low': round(lo, 3), 'CI_high': round(hi, 3), 'per_person_median': round(pp.median(), 3), 'n_pp': len(pp),
            'baseline_pooled': round(auc(d['y'], d['base']), 3),
            'baseline_per_person': round(pb.median(), 3)}


def pred_path(name): return os.path.join(SAVE_DIR, 'predictions', name + '.csv')
def done(name): return os.path.exists(pred_path(name))
def load_pred(name): return pd.read_csv(pred_path(name), dtype={'id': str})
print("✅ evaluation helpers defined")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — E1 + E2: TRANSFER BETWEEN DATASETS AND WITHIN-DATASET REFERENCE
# Transfer: train on all source participants, predict all target participants.
# Within: participant-level 10-fold CV inside the dataset (same features and model).
# ════════════════════════════════════════════════════════
PLAN = []   # (source, target, fset, scaling, model)
for fset in ['harm', 'harm_exp']:                                   # E1: smoking <-> smoking
    for m in ['LR', 'RF', 'XGB', 'GRU']:
        for sc in (['source', 'per_dataset'] if m != 'GRU' else ['per_dataset']):
            PLAN += [('SF', 'EM', fset, sc, m), ('EM', 'SF', fset, sc, m)]
        PLAN += [('EM', 'EM', fset, 'within', m), ('SF', 'SF', fset, 'within', m)]
for m in ['LR', 'XGB', 'GRU']:                                     # E2: common core, all pairs
    for s in DATA:
        for t in DATA:
            PLAN.append((s, t, 'core', 'within' if s == t else 'per_dataset', m))
PLAN = list(dict.fromkeys(PLAN))
print(f"{len(PLAN)} configurations")


def fit_any(m, seqs, nf):
    return train_gru(seqs, nf, EPOCHS) if m == 'GRU' else fit_flat(m, seqs)


def predict_any(m, model, seqs):
    return predict_gru(model, seqs) if m == 'GRU' else predict_flat(model, seqs)


t0 = time.time()
cache = {}   # trained source models, reused across targets
for (s, t, fset, sc, m) in PLAN:
    name = f"E_{s}_to_{t}__{fset}__{sc}__{m}"
    if done(name): continue
    feats = FSETS[fset]; nf = len(feats); set_seed(SEED)
    if s == t:                                                       # within-dataset CV
        allseq = seqs_of(DATA[s], feats); rows = []
        for k, (tr, te) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(allseq)):
            set_seed(SEED + k)
            train = [allseq[i] for i in tr]; test = [allseq[i] for i in te]
            mu, sd = stats_of(train)
            model = fit_any(m, standardise(train, mu, sd), nf)
            rows += to_rows(predict_any(m, model, standardise(test, mu, sd)), fold=k)
    else:
        key = (s, fset, m)
        src = seqs_of(DATA[s], feats); mu_s, sd_s = stats_of(src)
        if key not in cache:
            cache[key] = fit_any(m, standardise(src, mu_s, sd_s), nf)
        tgt = seqs_of(DATA[t], feats)
        mu_t, sd_t = (mu_s, sd_s) if sc == 'source' else stats_of(tgt)
        rows = to_rows(predict_any(m, cache[key], standardise(tgt, mu_t, sd_t)))
    pd.DataFrame(rows).to_csv(pred_path(name), index=False)
    print(f"  {name:45s} {(time.time()-t0)/60:6.1f} min")
print("✅ E1/E2 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — E3: DOES THE LARGE DATASET HELP SMALL STUDIES?
# Target EM (harmonised + engineered) and AL (core). Outer: participant-level CV on the target.
# In each fold, n target participants are sampled from the training folds (E3_REPEATS repeats).
# Conditions: target_only | source_only | pooled (source + target rows) | finetune (GRU pretrained on SF)
# ════════════════════════════════════════════════════════
E3_TARGETS = {'EM': 'harm_exp', 'AL': 'core'}
t0 = time.time()
for tgt, fset in E3_TARGETS.items():
    feats = FSETS[fset]; nf = len(feats)
    src = seqs_of(DATA['SF'], feats); mu_s, sd_s = stats_of(src); src_z = standardise(src, mu_s, sd_s)
    tseq = seqs_of(DATA[tgt], feats); mu_t, sd_t = stats_of(tseq); tz = standardise(tseq, mu_t, sd_t)
    # source models trained once (target standardised with its own feature statistics)
    src_models = {}
    for m in ['LR', 'XGB', 'GRU']:
        name0 = f"E3_{tgt}__source_only__{m}"
        if not done(name0):
            set_seed(SEED); src_models[m] = fit_any(m, src_z, nf)
            pd.DataFrame(to_rows(predict_any(m, src_models[m], tz))).to_csv(pred_path(name0), index=False)
    pre_path = os.path.join(SAVE_DIR, f"gru_pretrained_SF_{fset}.pt")
    if not os.path.exists(pre_path):
        set_seed(SEED); torch.save(train_gru(src_z, nf, EPOCHS).state_dict(), pre_path)
    pretrained = torch.load(pre_path, map_location=device)
    folds = list(KFold(E3_FOLDS, shuffle=True, random_state=SEED).split(tz))
    for n in E3_N[tgt]:
        for r in range(E3_REPEATS):
            name = f"E3_{tgt}__n{n:03d}__r{r}"
            if done(name): continue
            rows = []
            for k, (tr, te) in enumerate(folds):
                rng = np.random.default_rng(SEED + 100 * r + k)
                pick = rng.choice(tr, min(n, len(tr)), replace=False)
                train = [tz[i] for i in pick]; test = [tz[i] for i in te]
                set_seed(SEED + 100 * r + k)
                for m in ['LR', 'XGB']:
                    rows += to_rows(predict_flat(fit_flat(m, train), test), cond='target_only', model=m, fold=k)
                    rows += to_rows(predict_flat(fit_flat(m, src_z + train), test), cond='pooled', model=m, fold=k)
                rows += to_rows(predict_gru(train_gru(train, nf, EPOCHS, bs=8), test),
                                cond='target_only', model='GRU', fold=k)
                rows += to_rows(predict_gru(train_gru(train, nf, FT_EPOCHS, lr=3e-4, init=pretrained, bs=8), test),
                                cond='finetune', model='GRU', fold=k)
            pd.DataFrame(rows).assign(n=n, repeat=r).to_csv(pred_path(name), index=False)
            print(f"  {name:28s} {(time.time()-t0)/60:6.1f} min")
print("✅ E3 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — RESULTS
# ════════════════════════════════════════════════════════
import glob, matplotlib.pyplot as plt

# E1 / E2 table
rows = []
for (s, t, fset, sc, m) in PLAN:
    name = f"E_{s}_to_{t}__{fset}__{sc}__{m}"
    r = summarise_preds(load_pred(name), name)
    rows.append(dict(source=s, target=t, features=fset, scaling=sc, model=m, **{k: v for k, v in r.items() if k != 'config'}))
E12 = pd.DataFrame(rows)
E12.to_csv(os.path.join(SAVE_DIR, 'E1_E2_results.csv'), index=False)
print("E1 — smoking to smoking (harmonised items)")
print(E12[E12.features != 'core'].drop(columns=['features']).to_string(index=False))

# paired comparison: transfer vs within-target model, on the same target participants
cmp = []
for (s, t, fset, sc, m) in PLAN:
    if s == t: continue
    a = load_pred(f"E_{s}_to_{t}__{fset}__{sc}__{m}"); b = load_pred(f"E_{t}_to_{t}__{fset}__within__{m}")
    j = a.merge(b, on=['id', 'y', 'base'], suffixes=('', '_w'), how='inner') if len(a) == len(b) else None
    if j is None or len(j) != len(a):        # align by id and position when merge keys repeat
        a = a.assign(pos=a.groupby('id').cumcount()); b = b.assign(pos=b.groupby('id').cumcount())
        j = a.merge(b[['id', 'pos', 'p']], on=['id', 'pos'], suffixes=('', '_w'))
    lo, hi = boot_pooled(j, cols=('p', 'p_w'))
    pa, pw = per_person(j), per_person(j, 'p_w'); common = pa.index.intersection(pw.index)
    d = (pa[common] - pw[common]).to_numpy()
    cmp.append(dict(source=s, target=t, features=fset, scaling=sc, model=m,
                    transfer_minus_within=round(auc(j['y'], j['p']) - auc(j['y'], j['p_w']), 3),
                    CI_low=round(lo, 3), CI_high=round(hi, 3), within_person_median_diff=round(float(np.median(d)), 3),
                    share_transfer_better=round(float((d > 0).mean()), 2),
                    wilcoxon_p=stats.wilcoxon(d).pvalue if len(d) > 1 and np.any(d != 0) else np.nan))
CMP = pd.DataFrame(cmp)
def holm(p):
    p = np.asarray(p, float); ok = ~np.isnan(p); out = np.full_like(p, np.nan)
    if ok.any():
        q = p[ok]; o = np.argsort(q); m = len(q)
        adj = np.maximum.accumulate((m - np.arange(m)) * q[o]).clip(max=1)
        r = np.empty(m); r[o] = adj; out[ok] = r
    return out
# Holm correction within each family (feature set x model), as in the replication notebooks
CMP['wilcoxon_p_holm'] = CMP.groupby(['features', 'model'])['wilcoxon_p'].transform(holm)
CMP.to_csv(os.path.join(SAVE_DIR, 'transfer_vs_within.csv'), index=False)
print("\nTransfer minus within-target model (same target participants)")
print(CMP.round(4).to_string(index=False))

print("\nE2 — common core, pooled AUC (rows = source, columns = target; diagonal = within-dataset CV)")
for m in ['LR', 'XGB', 'GRU']:
    sub = E12[(E12.features == 'core') & (E12.model == m)]
    print(f"\n{m}:\n" + sub.pivot(index='source', columns='target', values='pooled_AUC').to_string())
    print("per-person median:\n" + sub.pivot(index='source', columns='target', values='per_person_median').to_string())

# E3 learning curves
e3 = []
for tgt in E3_TARGETS:
    so = {m: load_pred(f"E3_{tgt}__source_only__{m}") for m in ['LR', 'XGB', 'GRU']}
    for f in sorted(glob.glob(pred_path(f"E3_{tgt}__n*"))):
        d = pd.read_csv(f, dtype={'id': str}); n, r = int(d['n'].iloc[0]), int(d['repeat'].iloc[0])
        for (cond, m), g in d.groupby(['cond', 'model']):
            e3.append(dict(target=tgt, n=n, repeat=r, cond=cond, model=m, pooled=auc(g['y'], g['p']),
                           per_person=per_person(g).median()))
        e3.append(dict(target=tgt, n=n, repeat=r, cond='person_baseline', model='-', pooled=auc(d['y'], d['base']),
                       per_person=per_person(d, 'base').median()))
    for m, g in so.items():
        e3.append(dict(target=tgt, n=0, repeat=0, cond='source_only', model=m, pooled=auc(g['y'], g['p']),
                       per_person=per_person(g).median()))
E3 = pd.DataFrame(e3)
E3s = E3.groupby(['target', 'cond', 'model', 'n'])[['pooled', 'per_person']].agg(['mean', 'std']).round(3)
E3s.to_csv(os.path.join(SAVE_DIR, 'E3_learning_curves.csv'))
print("\nE3 — learning curves (mean and SD over repeats)"); print(E3s.to_string())

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex='col')
for c, tgt in enumerate(E3_TARGETS):
    for rr, metric in enumerate(['pooled', 'per_person']):
        ax = axes[rr, c]
        for (cond, m), g in E3[(E3.target == tgt) & (E3.n > 0)].groupby(['cond', 'model']):
            if cond == 'person_baseline' and m != '-': continue
            s = g.groupby('n')[metric].mean()
            ax.plot(s.index, s.values, marker='o', label=f"{cond} {m}".strip(' -'))
        for m, g in E3[(E3.target == tgt) & (E3.cond == 'source_only')].groupby('model'):
            ax.axhline(g[metric].mean(), ls=':', lw=1, color='grey')
        ax.set_title(f"{tgt}: {'pooled AUC' if metric == 'pooled' else 'per-participant median AUC'}")
        ax.set_xlabel('target participants used for training'); ax.axhline(0.5, color='k', lw=0.5)
axes[0, 1].legend(fontsize=7, loc='lower right')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'E3_learning_curves.png'), dpi=150); plt.show()
print("Dotted grey lines = source-only models (no target data).")


---
## Part 2 — Does the transfer come from lapse history? (added 8 Oct 2026)

Part 1 suggested that what transfers is **lapse history** (prior lapse, lapse lags, days since the last lapse).
Part 2 tests this directly by rerunning E1 and E2 with three feature conditions on the **same participants**:

| Condition | E1 (smoking ↔ smoking) | E2 (all pairs) |
|---|---|---|
| **full** (Part 1) | `harm` 23, `harm_exp` 36 | `core` 17 |
| **no_history** — lapse-history features removed | `harm_nohist` 22, `harm_exp_nohist` 32 | `core_nohist` 13 |
| **history_only** — only lapse history | — | `hist_only` 4 |

Removed lapse-history features: `prior_lapse`, `lapse_lag2`, `lapse_lag3`, `days_since_last_lapse`.

**Reading the result**
- If transfer is close to within-study performance with history but falls clearly below it without history,
  the transferable part is lapse history.
- If `history_only` transfers about as well as `full`, history alone carries the transfer.
- If `no_history` still transfers, the momentary signal (craving, feelings, context) generalises too.

**After a disconnect:** reconnect, then **Run all**. Cells 1–4 reload the data (about 1 minute);
Cells 5–6 skip everything already saved on Drive; Cell 8 resumes from the first configuration not yet saved.
Only cells 1–4 and 8–9 are needed for Part 2.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — PART 2: TRANSFER WITHOUT LAPSE HISTORY, AND WITH HISTORY ONLY
# Every configuration is saved to SAVE_DIR/predictions on Drive as soon as it finishes.
# ════════════════════════════════════════════════════════
HISTORY = ['prior_lapse', 'lapse_lag2', 'lapse_lag3', 'days_since_last_lapse']
FSETS['harm_nohist']     = [f for f in HARM_ITEMS if f not in HISTORY]
FSETS['harm_exp_nohist'] = [f for f in HARM_ITEMS + EXTRA if f not in HISTORY]
FSETS['core_nohist']     = [f for f in CORE if f not in HISTORY]
FSETS['hist_only']       = HISTORY
for k in ['harm_nohist', 'harm_exp_nohist', 'core_nohist', 'hist_only']:
    print(f"{k:16s} {len(FSETS[k]):2d} features")

NH_MODELS_E1 = ['LR', 'RF', 'XGB', 'GRU']   # drop 'RF' to save about a third of the run time
NH_MODELS_E2 = ['LR', 'XGB', 'GRU']

PLAN2 = []
for fset in ['harm_nohist', 'harm_exp_nohist']:                      # E1 without history
    for m in NH_MODELS_E1:
        for sc in (['source', 'per_dataset'] if m != 'GRU' else ['per_dataset']):
            PLAN2 += [('SF', 'EM', fset, sc, m), ('EM', 'SF', fset, sc, m)]
        PLAN2 += [('EM', 'EM', fset, 'within', m), ('SF', 'SF', fset, 'within', m)]
for fset in ['core_nohist', 'hist_only']:                            # E2 without history / history only
    for m in NH_MODELS_E2:
        for s in DATA:
            for t in DATA:
                PLAN2.append((s, t, fset, 'within' if s == t else 'per_dataset', m))
PLAN2 = list(dict.fromkeys(PLAN2))
todo = [c for c in PLAN2 if not done(f"E_{c[0]}_to_{c[1]}__{c[2]}__{c[3]}__{c[4]}")]
print(f"\n{len(PLAN2)} configurations, {len(PLAN2) - len(todo)} already saved, {len(todo)} to run -> {SAVE_DIR}")


def fit_any(m, seqs, nf):          # repeated from Cell 5 so Part 2 does not need Cell 5
    return train_gru(seqs, nf, EPOCHS) if m == 'GRU' else fit_flat(m, seqs)


def predict_any(m, model, seqs):
    return predict_gru(model, seqs) if m == 'GRU' else predict_flat(model, seqs)


t0 = time.time(); cache2 = {}
for (s, t, fset, sc, m) in todo:
    name = f"E_{s}_to_{t}__{fset}__{sc}__{m}"
    feats = FSETS[fset]; nf = len(feats); set_seed(SEED)
    if s == t:
        allseq = seqs_of(DATA[s], feats); rows = []
        for k, (tr, te) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(allseq)):
            set_seed(SEED + k)
            train = [allseq[i] for i in tr]; test = [allseq[i] for i in te]
            mu, sd = stats_of(train)
            model = fit_any(m, standardise(train, mu, sd), nf)
            rows += to_rows(predict_any(m, model, standardise(test, mu, sd)), fold=k)
    else:
        key = (s, fset, m)
        src = seqs_of(DATA[s], feats); mu_s, sd_s = stats_of(src)
        if key not in cache2:
            cache2[key] = fit_any(m, standardise(src, mu_s, sd_s), nf)
        tgt = seqs_of(DATA[t], feats)
        mu_t, sd_t = (mu_s, sd_s) if sc == 'source' else stats_of(tgt)
        rows = to_rows(predict_any(m, cache2[key], standardise(tgt, mu_t, sd_t)))
    tmp = pred_path(name) + '.tmp'                                   # write-then-rename: no half-written files
    pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))
    print(f"  {name:52s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 2 runs done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 9 — PART 2 RESULTS: FULL vs NO_HISTORY vs HISTORY_ONLY
# For each source -> target pair and model: transfer AUC, within-target AUC, and the transfer gap
# (transfer minus within) in each condition, plus a paired bootstrap of how the gap changes
# when history is removed (same target participants, same entries).
# ════════════════════════════════════════════════════════
def holm(p):
    p = np.asarray(p, float); ok = ~np.isnan(p); out = np.full_like(p, np.nan)
    if ok.any():
        q = p[ok]; o = np.argsort(q); m = len(q)
        adj = np.maximum.accumulate((m - np.arange(m)) * q[o]).clip(max=1)
        r = np.empty(m); r[o] = adj; out[ok] = r
    return out


def aligned(names):
    """Loads several prediction files for the same target and aligns them row by row (id + position)."""
    out = None
    for j, nm in enumerate(names):
        d = load_pred(nm).sort_values('id', kind='stable')
        d = d.assign(pos=d.groupby('id').cumcount())[['id', 'pos', 'y', 'base', 'p']].rename(columns={'p': f'p{j}'})
        out = d if out is None else out.merge(d[['id', 'pos', f'p{j}']], on=['id', 'pos'])
    return out


def boot_gap_change(j, n_boot=N_BOOT, seed=SEED):
    """CI of (transfer - within)[no_history] - (transfer - within)[full]; p0/p1 = full, p2/p3 = no_history."""
    rng = np.random.default_rng(seed); ids = j['id'].unique(); parts = {i: g for i, g in j.groupby('id')}
    vals = []
    for _ in range(n_boot):
        s = pd.concat([parts[i] for i in rng.choice(ids, len(ids))])
        a = [auc(s['y'], s[f'p{k}']) for k in range(4)]
        vals.append((a[2] - a[3]) - (a[0] - a[1]))
    return np.nanpercentile(vals, [2.5, 97.5])


PAIRS2 = []   # (source, target, full fset, no_history fset, scaling, model)
for full, nh in [('harm', 'harm_nohist'), ('harm_exp', 'harm_exp_nohist')]:
    for m in NH_MODELS_E1:
        for sc in (['source', 'per_dataset'] if m != 'GRU' else ['per_dataset']):
            PAIRS2 += [('SF', 'EM', full, nh, sc, m), ('EM', 'SF', full, nh, sc, m)]
for m in NH_MODELS_E2:
    for s in DATA:
        for t in DATA:
            if s != t: PAIRS2.append((s, t, 'core', 'core_nohist', 'per_dataset', m))

rows = []
for (s, t, full, nh, sc, m) in PAIRS2:
    names = [f"E_{s}_to_{t}__{full}__{sc}__{m}", f"E_{t}_to_{t}__{full}__within__{m}",
             f"E_{s}_to_{t}__{nh}__{sc}__{m}",   f"E_{t}_to_{t}__{nh}__within__{m}"]
    hist = [f"E_{s}_to_{t}__hist_only__{sc}__{m}", f"E_{t}_to_{t}__hist_only__within__{m}"]
    if full == 'core' and all(done(n) for n in hist):    # history_only is optional: reported when finished
        names += hist
    if not all(done(n) for n in names):
        print("  missing, skipped:", [n for n in names if not done(n)][:2]); continue
    j = aligned(names)
    A = [auc(j['y'], j[f'p{k}']) for k in range(len(names))]
    PP = [per_person(j, f'p{k}') for k in range(len(names))]
    lo, hi = boot_gap_change(j)
    # within-person: change in the per-participant transfer gap when history is removed
    common = PP[0].index.intersection(PP[1].index).intersection(PP[2].index).intersection(PP[3].index)
    dgap = ((PP[2][common] - PP[3][common]) - (PP[0][common] - PP[1][common])).to_numpy()
    r = dict(source=s, target=t, features=full, scaling=sc, model=m, n_people=j['id'].nunique(),
             baseline_pooled=round(auc(j['y'], j['base']), 3),
             full_transfer=round(A[0], 3), full_within=round(A[1], 3), full_gap=round(A[0] - A[1], 3),
             nohist_transfer=round(A[2], 3), nohist_within=round(A[3], 3), nohist_gap=round(A[2] - A[3], 3),
             gap_change=round((A[2] - A[3]) - (A[0] - A[1]), 3), gap_change_CI_low=round(lo, 3),
             gap_change_CI_high=round(hi, 3),
             pp_full_transfer=round(PP[0].median(), 3), pp_full_within=round(PP[1].median(), 3),
             pp_nohist_transfer=round(PP[2].median(), 3), pp_nohist_within=round(PP[3].median(), 3),
             pp_gap_change_median=round(float(np.median(dgap)), 3) if len(dgap) else np.nan,
             pp_wilcoxon_p=stats.wilcoxon(dgap).pvalue if len(dgap) > 1 and np.any(dgap != 0) else np.nan)
    if len(names) == 6:
        r.update(histonly_transfer=round(A[4], 3), histonly_within=round(A[5], 3),
                 histonly_gap=round(A[4] - A[5], 3),
                 pp_histonly_transfer=round(PP[4].median(), 3), pp_histonly_within=round(PP[5].median(), 3))
    rows.append(r)

P2 = pd.DataFrame(rows)
P2['pp_wilcoxon_p_holm'] = P2.groupby(['features', 'model'])['pp_wilcoxon_p'].transform(holm)
P2.to_csv(os.path.join(SAVE_DIR, 'part2_history_ablation.csv'), index=False)

pd.set_option('display.width', 250)
show = ['source', 'target', 'scaling', 'model', 'baseline_pooled', 'full_transfer', 'full_within', 'full_gap',
        'nohist_transfer', 'nohist_within', 'nohist_gap', 'gap_change', 'gap_change_CI_low', 'gap_change_CI_high']
for feats in ['harm', 'harm_exp', 'core']:
    sub = P2[P2.features == feats]
    if sub.empty: continue
    print(f"\n=== {feats}: pooled AUC — full vs no_history (gap = transfer − within; gap_change < 0 = transfer "
          f"loses more without history) ===")
    print(sub[show].to_string(index=False))
    print(f"\n--- {feats}: per-participant median AUC ---")
    print(sub[['source', 'target', 'scaling', 'model', 'pp_full_transfer', 'pp_full_within', 'pp_nohist_transfer',
               'pp_nohist_within', 'pp_gap_change_median', 'pp_wilcoxon_p', 'pp_wilcoxon_p_holm']]
          .round(4).to_string(index=False))

core = P2[P2.features == 'core']
if not core.empty and 'histonly_transfer' in core:
    print("\n=== E2 common core: transfer gap in each condition (pooled AUC) ===")
    print(core[['source', 'target', 'model', 'baseline_pooled', 'full_gap', 'nohist_gap', 'histonly_gap',
                'full_transfer', 'nohist_transfer', 'histonly_transfer']].to_string(index=False))

# matrices for the paper: pooled and per-participant AUC per condition (rows = source, columns = target)
mats = []
for fset in ['core', 'core_nohist', 'hist_only']:
    for m in NH_MODELS_E2:
        for s in DATA:
            for t in DATA:
                nm = f"E_{s}_to_{t}__{fset}__{'within' if s == t else 'per_dataset'}__{m}"
                if done(nm):
                    d = load_pred(nm)
                    mats.append(dict(features=fset, model=m, source=s, target=t, pooled=round(auc(d['y'], d['p']), 3),
                                     per_person=round(per_person(d).median(), 3)))
MATS = pd.DataFrame(mats)
MATS.to_csv(os.path.join(SAVE_DIR, 'part2_E2_matrices.csv'), index=False)
for m in NH_MODELS_E2:
    for fset in ['core', 'core_nohist', 'hist_only']:
        sub = MATS[(MATS.model == m) & (MATS.features == fset)]
        if sub.empty: continue
        print(f"\n{m} — {fset}: pooled | per-participant median (rows = source, columns = target)")
        print(pd.concat([sub.pivot(index='source', columns='target', values='pooled'),
                         sub.pivot(index='source', columns='target', values='per_person')],
                        axis=1, keys=['pooled', 'per_person']).to_string())
print(f"\nSaved: {SAVE_DIR}/part2_history_ablation.csv and part2_E2_matrices.csv")


---
## Part 3 — Is timing what ties the momentary signal to the sampling design? (added 9 Oct 2026)

Part 2 showed that, without lapse history, models transfer between the two prompted studies (EMASENS ↔ alcohol)
but not to or from Smoke Free. The 13 remaining features mix two kinds of information:

| Set | Features | n |
|---|---|---|
| `craving_only` | craving, severity_change, rolling3_sev_mean, rolling3_sev_std, sev_lag1–3 | 7 |
| `timing_only` | mins_since_prev, days_since_start, log_days_since_qd, hour_sin, hour_cos, is_weekend | 6 |
| `craving_now` | craving at the current entry only | 1 |

Timing means different things in the two designs: in Smoke Free people open the app when they want to, so the gap
since the last entry and the time of day reflect behaviour; in prompted studies they mostly reflect the prompt schedule.

**Reading the result**
- `craving_only` transfers to and from Smoke Free (gap near 0) while `timing_only` does not → **timing** carries the
  design-specific part, and craving dynamics generalise across designs.
- Neither transfers to or from Smoke Free → the difference is in **craving itself** (a self-initiated craving report
  is not the same measurement as a prompted rating).
- `craving_now` is a single feature, so a logistic model can only get the direction right or wrong.
- Cell 11 also prints, for every feature and dataset, the direction of its relation with the next lapse (one feature at a
  time, both across all entries and within person). `SF_differs = True` flags features where the two prompted studies
  agree and Smoke Free goes the other way — a direct, model-free explanation of failed transfer.

LR and XGBoost only (about 20 minutes on Colab). Set `P3_MODELS = ['LR', 'XGB', 'GRU']` to add GRU (about 2 more hours).
Like Part 2, every configuration is saved to Drive as it finishes; after a disconnect run Cells 1–4 and then 10–11.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 10 — PART 3: CRAVING vs TIMING (no lapse history in any set)
# ════════════════════════════════════════════════════════
CRAVING = ['craving', 'severity_change', 'rolling3_sev_mean', 'rolling3_sev_std', 'sev_lag1', 'sev_lag2', 'sev_lag3']
TIMING  = ['mins_since_prev', 'days_since_start', 'log_days_since_qd', 'hour_sin', 'hour_cos', 'is_weekend']
assert sorted(CRAVING + TIMING) == sorted(f for f in CORE if f not in
       ['prior_lapse', 'lapse_lag2', 'lapse_lag3', 'days_since_last_lapse']), "CRAVING + TIMING must equal core_nohist"
FSETS['craving_only'] = CRAVING
FSETS['timing_only']  = TIMING
FSETS['craving_now']  = ['craving']
P3_SETS   = ['craving_only', 'timing_only', 'craving_now']
P3_MODELS = ['LR', 'XGB']            # add 'GRU' for the full set (slow)


def fit_any(m, seqs, nf):            # repeated so Part 3 does not need Cells 5 or 8
    return train_gru(seqs, nf, EPOCHS) if m == 'GRU' else fit_flat(m, seqs)


def predict_any(m, model, seqs):
    return predict_gru(model, seqs) if m == 'GRU' else predict_flat(model, seqs)


PLAN3 = [(s, t, fset, 'within' if s == t else 'per_dataset', m)
         for fset in P3_SETS for m in P3_MODELS for s in DATA for t in DATA]
todo3 = [c for c in PLAN3 if not done(f"E_{c[0]}_to_{c[1]}__{c[2]}__{c[3]}__{c[4]}")]
print(f"{len(PLAN3)} configurations, {len(PLAN3) - len(todo3)} already saved, {len(todo3)} to run -> {SAVE_DIR}")

t0 = time.time(); cache3 = {}
for (s, t, fset, sc, m) in todo3:
    name = f"E_{s}_to_{t}__{fset}__{sc}__{m}"
    feats = FSETS[fset]; nf = len(feats); set_seed(SEED)
    if s == t:
        allseq = seqs_of(DATA[s], feats); rows = []
        for k, (tr, te) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(allseq)):
            set_seed(SEED + k)
            train = [allseq[i] for i in tr]; test = [allseq[i] for i in te]
            mu, sd = stats_of(train)
            model = fit_any(m, standardise(train, mu, sd), nf)
            rows += to_rows(predict_any(m, model, standardise(test, mu, sd)), fold=k)
    else:
        key = (s, fset, m)
        src = seqs_of(DATA[s], feats); mu_s, sd_s = stats_of(src)
        if key not in cache3:
            cache3[key] = fit_any(m, standardise(src, mu_s, sd_s), nf)
        tgt = seqs_of(DATA[t], feats); mu_t, sd_t = stats_of(tgt)
        rows = to_rows(predict_any(m, cache3[key], standardise(tgt, mu_t, sd_t)))
    tmp = pred_path(name) + '.tmp'
    pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))
    print(f"  {name:48s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 3 runs done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 11 — PART 3 RESULTS
# (a) transfer gap (transfer − within, same target entries) for core_nohist, craving_only, timing_only, craving_now,
#     with a participant-level bootstrap CI for each gap;
# (b) source × target matrices (pooled and per-participant median AUC);
# (c) standardised logistic-regression coefficients fitted in each dataset (same sign everywhere?).
# ════════════════════════════════════════════════════════
def aligned(names):
    out = None
    for j, nm in enumerate(names):
        d = load_pred(nm).sort_values('id', kind='stable')
        d = d.assign(pos=d.groupby('id').cumcount())[['id', 'pos', 'y', 'base', 'p']].rename(columns={'p': f'p{j}'})
        out = d if out is None else out.merge(d[['id', 'pos', f'p{j}']], on=['id', 'pos'])
    return out


SETS3 = ['core_nohist'] + P3_SETS
rows = []
for m in P3_MODELS:
    for s in DATA:
        for t in DATA:
            if s == t: continue
            r = dict(source=s, target=t, model=m,
                     pair_type='prompted↔prompted' if 'SF' not in (s, t) else 'involves Smoke Free')
            for fset in SETS3:
                a, b = f"E_{s}_to_{t}__{fset}__per_dataset__{m}", f"E_{t}_to_{t}__{fset}__within__{m}"
                if not (done(a) and done(b)): continue
                j = aligned([a, b])
                lo, hi = boot_pooled(j, cols=('p0', 'p1'))
                pa, pw = per_person(j, 'p0'), per_person(j, 'p1'); common = pa.index.intersection(pw.index)
                r.update({f'{fset}_transfer': round(auc(j['y'], j['p0']), 3),
                          f'{fset}_within': round(auc(j['y'], j['p1']), 3),
                          f'{fset}_gap': round(auc(j['y'], j['p0']) - auc(j['y'], j['p1']), 3),
                          f'{fset}_gap_CI': f"[{lo:.3f}, {hi:.3f}]",
                          f'{fset}_gap_CI_low': round(lo, 3), f'{fset}_gap_CI_high': round(hi, 3),
                          f'{fset}_pp_gap_median': round(float(np.median(pa[common] - pw[common])), 3)})
            rows.append(r)
P3 = pd.DataFrame(rows)
P3.to_csv(os.path.join(SAVE_DIR, 'part3_craving_vs_timing.csv'), index=False)

pd.set_option('display.width', 250)
for m in P3_MODELS:
    sub = P3[P3.model == m]
    print(f"\n=== {m}: transfer gap in pooled AUC (transfer − within; 95% bootstrap CI) ===")
    cols = ['source', 'target', 'pair_type'] + [c for f in SETS3 for c in (f'{f}_gap', f'{f}_gap_CI') if c in sub]
    print(sub[cols].to_string(index=False))
    print(f"\n--- {m}: transfer and within pooled AUC ---")
    cols = ['source', 'target'] + [c for f in SETS3 for c in (f'{f}_transfer', f'{f}_within') if c in sub]
    print(sub[cols].to_string(index=False))
    print(f"\n--- {m}: per-participant median of (transfer − within) ---")
    cols = ['source', 'target'] + [f'{f}_pp_gap_median' for f in SETS3 if f'{f}_pp_gap_median' in sub]
    print(sub[cols].to_string(index=False))

# summary: mean gap by pair type
summ = []
for m in P3_MODELS:
    for pt, g in P3[P3.model == m].groupby('pair_type'):
        summ.append({'model': m, 'pair_type': pt, **{f: round(g[f'{f}_gap'].mean(), 3) for f in SETS3 if f'{f}_gap' in g}})
print("\n=== Mean transfer gap by pair type (pooled AUC) ===")
print(pd.DataFrame(summ).to_string(index=False))

# matrices
mats = []
for fset in P3_SETS:
    for m in P3_MODELS:
        for s in DATA:
            for t in DATA:
                nm = f"E_{s}_to_{t}__{fset}__{'within' if s == t else 'per_dataset'}__{m}"
                if done(nm):
                    d = load_pred(nm)
                    mats.append(dict(features=fset, model=m, source=s, target=t,
                                     pooled=round(auc(d['y'], d['p']), 3), per_person=round(per_person(d).median(), 3)))
MATS3 = pd.DataFrame(mats)
MATS3.to_csv(os.path.join(SAVE_DIR, 'part3_matrices.csv'), index=False)
for m in P3_MODELS:
    for fset in P3_SETS:
        sub = MATS3[(MATS3.model == m) & (MATS3.features == fset)]
        if sub.empty: continue
        print(f"\n{m} — {fset}: pooled | per-participant median (rows = source, columns = target)")
        print(pd.concat([sub.pivot(index='source', columns='target', values='pooled'),
                         sub.pivot(index='source', columns='target', values='per_person')],
                        axis=1, keys=['pooled', 'per_person']).to_string())

# Direction of each feature's relation with the next-entry lapse, in each dataset.
# One feature at a time (univariable, standardised LR), because the craving features are collinear
# (severity_change = craving − sev_lag1), so multivariable coefficients cannot be compared by sign.
# 'pooled' = across all entries; 'within' = after centring the feature on each person's own mean,
# which shows the moment-to-moment direction inside a person.
coef = {}
for name in DATA:
    d0 = DATA[name]
    for fset in ['craving_only', 'timing_only']:
        for f in FSETS[fset]:
            for kind in ['pooled', 'within']:
                d1 = d0.copy()
                if kind == 'within':
                    d1[f] = d1[f] - d1.groupby('id')[f].transform('mean')
                sq = seqs_of(d1, [f]); mu, sd = stats_of(sq)
                clf = fit_flat('LR', standardise(sq, mu, sd))
                coef[(fset, f, kind, name)] = round(float(clf.coef_[0][0]), 3)
COEF = pd.Series(coef).rename_axis(['set', 'feature', 'kind', 'dataset']).unstack('dataset')[list(DATA)]
COEF['same_sign_all'] = np.sign(COEF[list(DATA)]).nunique(axis=1) == 1
COEF['SF_differs'] = (np.sign(COEF['SF']) != np.sign(COEF['EM'])) & (np.sign(COEF['EM']) == np.sign(COEF['AL']))
COEF.to_csv(os.path.join(SAVE_DIR, 'part3_feature_directions.csv'))
print("\n=== Direction of each feature's relation with the next lapse (univariable standardised LR coefficient) ===")
print("SF_differs = True: the two prompted studies agree and Smoke Free has the opposite sign")
print(COEF.to_string())
print(f"\nSaved: part3_craving_vs_timing.csv, part3_matrices.csv, part3_feature_directions.csv in {SAVE_DIR}")


---
## Part 4 — Fold-matched correction of all "transfer vs within" comparisons (added 10 Oct 2026)

**Why.** Each "within" model is evaluated by 10-fold cross-validation, and the pooled AUC was computed after stacking
the predictions of the 10 different fold models. Their intercepts and scaling differ (lapse rates differ between folds),
so stacking adds offset noise and pulls the pooled AUC down. The transfer model is a single model, so it has no such
penalty. The clearest evidence: with current craving as the only feature, a logistic model ranks entries identically
whatever data it was trained on, yet within-Smoke Free pooled AUC was 0.548 against 0.624 for transfer.

**Fix (no retraining).** Every within prediction file stores its fold. Transfer predictions on the same target are split
by the same participant folds. In each fold, AUC is computed for transfer and for within on exactly the same test
participants; the **fold-averaged AUC** and the **mean per-fold gap** (with a t-interval over folds) are reported.
E3 is corrected the same way (its 5 outer folds; source-only models are split by the same folds).

Per-participant AUCs are computed inside each person and were never affected.

Run Cells 1–4, then Cell 12 (a few minutes; nothing is retrained).


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 12 — PART 4: FOLD-MATCHED AUC FOR ALL COMPARISONS (Parts 1–3 and E3)
# ════════════════════════════════════════════════════════
import glob, re, matplotlib.pyplot as plt


def fold_aucs(d, col='p'):
    return pd.Series({k: auc(g['y'], g[col]) for k, g in d.groupby('fold')})


def t_ci(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return (x.mean() - h, x.mean() + h)


PART = {'harm': 'P1', 'harm_exp': 'P1', 'core': 'P1',
        'harm_nohist': 'P2', 'harm_exp_nohist': 'P2', 'core_nohist': 'P2', 'hist_only': 'P2',
        'craving_only': 'P3', 'timing_only': 'P3', 'craving_now': 'P3'}
pat = re.compile(r'^E_(\w+?)_to_(\w+?)__(.+?)__(source|per_dataset)__(\w+)$')
rows = []
for f in sorted(glob.glob(pred_path('E_*'))):
    mt = pat.match(os.path.basename(f)[:-4])
    if not mt: continue
    s, t, fset, sc, m = mt.groups()
    wname = f"E_{t}_to_{t}__{fset}__within__{m}"
    if not done(wname): continue
    w = load_pred(wname); tr = load_pred(os.path.basename(f)[:-4])
    fmap = w.groupby('id')['fold'].first()
    tr['fold'] = tr['id'].map(fmap)
    at, aw, ab = fold_aucs(tr), fold_aucs(w), fold_aucs(w, 'base')
    dk = (at - aw).dropna()
    lo, hi = t_ci(dk)
    rows.append(dict(part=PART.get(fset, '?'), source=s, target=t, features=fset, scaling=sc, model=m,
                     pooled_transfer=round(auc(tr['y'], tr['p']), 3), pooled_within=round(auc(w['y'], w['p']), 3),
                     pooled_gap=round(auc(tr['y'], tr['p']) - auc(w['y'], w['p']), 3),
                     fa_transfer=round(at.mean(), 3), fa_within=round(aw.mean(), 3), fa_baseline=round(ab.mean(), 3),
                     fa_gap=round(dk.mean(), 3), fa_gap_CI_low=round(lo, 3), fa_gap_CI_high=round(hi, 3),
                     folds_used=len(dk), folds_transfer_better=int((dk > 0).sum())))
FA = pd.DataFrame(rows)
FA['within_penalty'] = (FA['fa_within'] - FA['pooled_within']).round(3)      # how much stacking deflated "within"
FA['gap_shift'] = (FA['fa_gap'] - FA['pooled_gap']).round(3)
FA['CI'] = FA.apply(lambda r: f"[{r.fa_gap_CI_low:.3f}, {r.fa_gap_CI_high:.3f}]", axis=1)
FA.to_csv(os.path.join(SAVE_DIR, 'part4_fold_matched_gaps.csv'), index=False)
pd.set_option('display.width', 250)

print("=== How much did stacking fold predictions deflate the within-dataset pooled AUC? ===")
print("(within_penalty = fold-averaged − stacked; positive = stacking made 'within' look worse)")
pen = FA.drop_duplicates(['target', 'features', 'model'])
print(pen.groupby(['target', 'model'])['within_penalty'].agg(['mean', 'min', 'max']).round(3).to_string())

cols = ['source', 'target', 'scaling', 'model', 'pooled_gap', 'fa_transfer', 'fa_within', 'fa_gap', 'CI',
        'folds_used', 'folds_transfer_better']
for part, title in [('P1', 'PART 1 (full features)'), ('P2', 'PART 2 (history ablation)'),
                    ('P3', 'PART 3 (craving vs timing)')]:
    sub = FA[FA.part == part]
    if sub.empty: continue
    print(f"\n\n######## {title}: fold-matched gap (transfer − within, mean over folds, 95% t-interval) ########")
    for fset in dict.fromkeys(sub.features):
        print(f"\n--- {fset} ---")
        print(sub[sub.features == fset][cols].to_string(index=False))

# Compact E2 / Part 2–3 summary: fold-matched gap per pair and feature set (per_dataset scaling)
e2sets = [f for f in ['core', 'core_nohist', 'hist_only', 'craving_only', 'timing_only', 'craving_now'] if f in set(FA.features)]
sub = FA[(FA.features.isin(e2sets)) & (FA.scaling == 'per_dataset')]
if not sub.empty:
    print("\n\n######## Common-core pairs: fold-matched gap by feature set (rows = pair) ########")
    for m in sorted(sub.model.unique()):
        piv = sub[sub.model == m].pivot_table(index=['source', 'target'], columns='features', values='fa_gap')
        print(f"\n{m}:\n" + piv[[c for c in e2sets if c in piv]].round(3).to_string())
    sub = sub.assign(pair_type=np.where((sub.source != 'SF') & (sub.target != 'SF'), 'prompted↔prompted',
                                        'involves Smoke Free'))
    print("\nMean fold-matched gap by pair type:")
    print(sub.pivot_table(index=['model', 'pair_type'], columns='features', values='fa_gap')
          [[c for c in e2sets if c in set(sub.features)]].round(3).to_string())

# ── E3: fold-averaged learning curves ────────────────────────────────────────
e3 = []
for tgt in ['EM', 'AL']:
    files = sorted(glob.glob(pred_path(f"E3_{tgt}__n*")))
    if not files: continue
    fmap = pd.read_csv(files[0], dtype={'id': str}).groupby('id')['fold'].first()
    for f in files:
        d = pd.read_csv(f, dtype={'id': str}); n, r = int(d['n'].iloc[0]), int(d['repeat'].iloc[0])
        for (cond, m), g in d.groupby(['cond', 'model']):
            e3.append(dict(target=tgt, n=n, repeat=r, cond=cond, model=m, fa_auc=fold_aucs(g).mean(),
                           stacked_auc=auc(g['y'], g['p'])))
        g = d[(d.cond == 'target_only') & (d.model == 'LR')]
        e3.append(dict(target=tgt, n=n, repeat=r, cond='person_baseline', model='-',
                       fa_auc=fold_aucs(g, 'base').mean(), stacked_auc=auc(g['y'], g['base'])))
    for m in ['LR', 'XGB', 'GRU']:
        nm = f"E3_{tgt}__source_only__{m}"
        if done(nm):
            d = load_pred(nm); d['fold'] = d['id'].map(fmap)
            e3.append(dict(target=tgt, n=0, repeat=0, cond='source_only', model=m, fa_auc=fold_aucs(d).mean(),
                           stacked_auc=auc(d['y'], d['p'])))
if e3:
    E3F = pd.DataFrame(e3)
    E3Fs = E3F.groupby(['target', 'cond', 'model', 'n'])[['fa_auc', 'stacked_auc']].agg(['mean', 'std']).round(3)
    E3Fs.to_csv(os.path.join(SAVE_DIR, 'part4_E3_fold_averaged.csv'))
    print("\n\n######## E3 learning curves: fold-averaged pooled AUC (fa) vs stacked (as reported before) ########")
    print(E3Fs.to_string())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, tgt in zip(axes, ['EM', 'AL']):
        g0 = E3F[E3F.target == tgt]
        for (cond, m), g in g0[g0.n > 0].groupby(['cond', 'model']):
            sr = g.groupby('n')['fa_auc'].mean()
            ax.plot(sr.index, sr.values, marker='o', label=f"{cond} {m}".strip(' -'))
        for m, g in g0[g0.cond == 'source_only'].groupby('model'):
            ax.axhline(g['fa_auc'].mean(), ls=':', lw=1, color='grey')
        ax.set_title(f"{tgt}: fold-averaged pooled AUC"); ax.set_xlabel('target participants used for training')
    axes[1].legend(fontsize=7, loc='lower right')
    plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'part4_E3_fold_averaged.png'), dpi=150); plt.show()

print(f"\nSaved: part4_fold_matched_gaps.csv, part4_E3_fold_averaged.csv/.png in {SAVE_DIR}")


---
## Part 5 — Tables and figures for the manuscript (added 15 Oct 2026)

Cell 13 needs only Cells 1–4 and the saved results from Parts 1–4 (nothing is retrained). It produces:

- **Table 1** — the three datasets as analysed (participants, entries, lapse rate, participants with both outcomes,
  sampling design and timing). Counts are given both per entry and per next-entry pair (the unit the models predict).
- **Figure 1** — fold-matched transfer gap by feature set, split by pair type ("involves Smoke Free" vs
  "prompted ↔ prompted"); one dot per source → target pair, bar = mean. LR (main) and XGB/GRU (sensitivity).
- **Figure 2** — E3 learning curves (fold-averaged pooled AUC) with the person-level baseline and the
  source-only references labelled.
- **Table 2** — fold-matched gaps for the common-core feature sets, ready to paste (CSV).

Figures are saved as PNG (300 dpi) and PDF in `SAVE_DIR/figures`.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 13 — PART 5: TABLE 1, FIGURES 1–2, TABLE 2
# ════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
FIG_DIR = os.path.join(SAVE_DIR, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)
NAMES = {'SF': 'Smoke Free', 'EM': 'EMASENS', 'AL': 'Alcohol (Wyant et al.)'}
DESIGN = {'SF': ('Smoking', 'Self-initiated app entries (unprompted)'),
          'EM': ('Smoking', 'Prompted EMA, 16/day for 10 days'),
          'AL': ('Alcohol', 'Prompted EMA, 4/day for up to ~3 months')}

# ── Table 1 ──────────────────────────────────────────────────────────────────
rows = []
for k, d in DATA.items():
    d = d.sort_values(['id', 't'])
    g = d.groupby('id')
    n_ent, lap = g.size(), g['label'].sum()
    # next-entry pairs: features at entry t, outcome = lapse at entry t+1 (the modelled unit)
    nxt = d.assign(y_next=g['label'].shift(-1)).dropna(subset=['y_next'])
    gn = nxt.groupby('id')['y_next']
    both_next = ((gn.sum() > 0) & (gn.sum() < gn.size())).sum()
    gap = g['t'].diff().dt.total_seconds().div(60).dropna()
    days = g['t'].agg(lambda s: (s.max() - s.min()).total_seconds() / 86400).clip(lower=1)
    rows.append({
        'Dataset': NAMES[k], 'Substance': DESIGN[k][0], 'Sampling design': DESIGN[k][1],
        'Participants': d['id'].nunique(),
        'Entries (answered EMAs / app entries)': f"{len(d):,}",
        'Entries per participant, median (IQR)': f"{n_ent.median():.0f} ({n_ent.quantile(.25):.0f}–{n_ent.quantile(.75):.0f})",
        'Lapse entries, n (%)': f"{int(d['label'].sum()):,} ({100 * d['label'].mean():.1f}%)",
        'Participants with ≥1 lapse, n (%)': f"{(lap > 0).sum()} ({100 * (lap > 0).mean():.1f}%)",
        'Participants with both lapse and non-lapse entries, n (%)':
            f"{((lap > 0) & (lap < n_ent)).sum()} ({100 * ((lap > 0) & (lap < n_ent)).mean():.1f}%)",
        'Next-entry pairs analysed': f"{len(nxt):,}",
        'Lapse rate in next-entry outcome': f"{100 * nxt['y_next'].mean():.1f}%",
        'Participants with both outcomes in next-entry pairs (per-participant AUC computable), n (%)':
            f"{both_next} ({100 * both_next / d['id'].nunique():.1f}%)",
        'Time between entries, median (IQR), minutes':
            f"{gap.median():.0f} ({gap.quantile(.25):.0f}–{gap.quantile(.75):.0f})",
        'Irregularity of timing, SD of log(1 + minutes)': f"{np.log1p(gap).std():.2f}",
        'Entries per day of participation, median': f"{(n_ent / days).median():.1f}"})
T1 = pd.DataFrame(rows).set_index('Dataset').T
T1.to_csv(os.path.join(SAVE_DIR, 'table1_datasets.csv'))
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 60)
print("TABLE 1 — Datasets as analysed\n"); print(T1.to_string())

# ── Load fold-matched results (Part 4) ───────────────────────────────────────
FA = pd.read_csv(os.path.join(SAVE_DIR, 'part4_fold_matched_gaps.csv'))
FA['pair_type'] = np.where((FA.source != 'SF') & (FA.target != 'SF'), 'Prompted ↔ prompted', 'Involves Smoke Free')
SETS = [('core', 'All 17\nfeatures'), ('hist_only', 'Lapse history\nonly (4)'), ('core_nohist', 'No lapse\nhistory (13)'),
        ('craving_only', 'Craving over\nentries (7)'), ('timing_only', 'Timing\nonly (6)'), ('craving_now', 'Current\ncraving (1)')]

# ── Figure 1: transfer gap by feature set and pair type ──────────────────────
def figure1(model, fname):
    sub = FA[(FA.model == model) & (FA.scaling == 'per_dataset') & FA.features.isin([s for s, _ in SETS])]
    sets = [(s, l) for s, l in SETS if s in set(sub.features)]
    if sub.empty: return
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    cols = {'Involves Smoke Free': '#C0504D', 'Prompted ↔ prompted': '#4F81BD'}
    w = 0.36
    for j, (pt, off) in enumerate([('Involves Smoke Free', -w / 2), ('Prompted ↔ prompted', w / 2)]):
        for i, (fs, _) in enumerate(sets):
            v = sub[(sub.features == fs) & (sub.pair_type == pt)]['fa_gap'].to_numpy()
            if len(v) == 0: continue
            ax.bar(i + off, v.mean(), w * 0.9, color=cols[pt], alpha=0.35, edgecolor=cols[pt],
                   label=pt if i == 0 else None)
            ax.scatter(np.full(len(v), i + off) + np.linspace(-0.07, 0.07, len(v)), v, s=22, color=cols[pt], zorder=3,
                       marker='o' if pt == 'Involves Smoke Free' else '^', edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(range(len(sets))); ax.set_xticklabels([l for _, l in sets], fontsize=8.5)
    ax.set_ylabel('Transfer − within-target AUC\n(fold-matched)')
    ax.legend(frameon=False, fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2)
    ax.yaxis.grid(True, color='#E6E6E6', lw=0.6); ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    for ext in ('png', 'pdf'): plt.savefig(os.path.join(FIG_DIR, f'{fname}.{ext}'), dpi=300, bbox_inches='tight')
    plt.show()

figure1('LR', 'figure1_transfer_gap_by_feature_set_LR')
figure1('XGB', 'figureS1_transfer_gap_by_feature_set_XGB')
figure1('GRU', 'figureS2_transfer_gap_by_feature_set_GRU')

# ── Figure 2: E3 learning curves with baseline ───────────────────────────────
pth = os.path.join(SAVE_DIR, 'part4_E3_fold_averaged.csv')
if os.path.exists(pth):
    E3F = pd.read_csv(pth, header=[0, 1], index_col=[0, 1, 2, 3])
    E3F.columns = ['_'.join(c) for c in E3F.columns]; E3F = E3F.reset_index()
    E3F['n'] = E3F['n'].astype(int)
    STY = {('target_only', 'LR'): ('Target data only — LR', '#7F7F7F', '-', 'o'),
           ('target_only', 'GRU'): ('Target data only — GRU', '#BFBFBF', '-', 's'),
           ('pooled', 'LR'): ('Smoke Free + target pooled — LR', '#4F81BD', '-', 'o'),
           ('finetune', 'GRU'): ('Pretrained on Smoke Free, fine-tuned — GRU', '#C0504D', '-', 's')}
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    for ax, tgt in zip(axes, ['EM', 'AL']):
        g0 = E3F[E3F.target == tgt]
        for (cond, m), (lab, c, ls, mk) in STY.items():
            g = g0[(g0.cond == cond) & (g0.model == m) & (g0.n > 0)].sort_values('n')
            if g.empty: continue
            ax.errorbar(g.n, g.fa_auc_mean, yerr=g.fa_auc_std.fillna(0), color=c, ls=ls, marker=mk, ms=4,
                        capsize=2, lw=1.4, label=lab)
        for m, ls in [('LR', ':'), ('GRU', '-.')]:
            v = g0[(g0.cond == 'source_only') & (g0.model == m)]['fa_auc_mean']
            if len(v): ax.axhline(v.iloc[0], color='#4F81BD' if m == 'LR' else '#C0504D', ls=ls, lw=1,
                                  label=f'Smoke Free model, no target data — {m}')
        b = g0[g0.cond == 'person_baseline']['fa_auc_mean']
        if len(b): ax.axhline(b.iloc[0], color='k', ls='--', lw=1.2, label="Person baseline (running lapse rate)")
        ax.set_title(NAMES[tgt].split(' (')[0], fontsize=10)
        ax.set_xlabel('Target-study participants used for training'); ax.set_ylabel('Pooled AUC (fold-averaged)')
        ax.spines[['top', 'right']].set_visible(False)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=3, fontsize=7.5, frameon=False, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=(0, 0.12, 1, 1))
    for ext in ('png', 'pdf'): plt.savefig(os.path.join(FIG_DIR, f'figure2_small_study_learning_curves.{ext}'), dpi=300)
    plt.show()

# ── Table 2: fold-matched gaps, common core, all models ──────────────────────
T2 = FA[(FA.scaling == 'per_dataset') & FA.features.isin([s for s, _ in SETS])].copy()
T2['Pair'] = T2['source'].map(lambda x: NAMES[x].split(' (')[0]) + ' → ' + T2['target'].map(lambda x: NAMES[x].split(' (')[0])
T2['Gap [95% CI]'] = T2.apply(lambda r: f"{r.fa_gap:+.3f} [{r.fa_gap_CI_low:+.3f}, {r.fa_gap_CI_high:+.3f}]", axis=1)
T2['Folds transfer better'] = T2['folds_transfer_better'].astype(str) + '/' + T2['folds_used'].astype(str)
T2['Feature set'] = T2['features'].map({s: l.replace('\n', ' ') for s, l in SETS})
T2 = T2[['model', 'Feature set', 'Pair', 'pair_type', 'fa_transfer', 'fa_within', 'Gap [95% CI]', 'Folds transfer better']]
T2 = T2.rename(columns={'model': 'Model', 'pair_type': 'Pair type', 'fa_transfer': 'Transfer AUC', 'fa_within': 'Within AUC'})
order = {s: i for i, (s, _) in enumerate(SETS)}
T2 = T2.sort_values(['Model', 'Feature set', 'Pair type', 'Pair'],
                    key=lambda c: c.map({l.replace('\n', ' '): i for i, (_, l) in enumerate(SETS)}) if c.name == 'Feature set' else c)
T2.to_csv(os.path.join(SAVE_DIR, 'table2_fold_matched_gaps.csv'), index=False)
print("\n\nTABLE 2 — Fold-matched transfer gaps (LR shown; all models in table2_fold_matched_gaps.csv)\n")
print(T2[T2.Model == 'LR'].to_string(index=False))

summ = (FA[(FA.scaling == 'per_dataset') & FA.features.isin([s for s, _ in SETS])]
        .pivot_table(index=['model', 'pair_type'], columns='features', values='fa_gap')
        [[s for s, _ in SETS if s in set(FA.features)]].round(3))
summ.to_csv(os.path.join(SAVE_DIR, 'table2b_mean_gap_by_pair_type.csv'))
print("\nMean fold-matched gap by pair type (Figure 1 bars):\n"); print(summ.to_string())
print(f"\nSaved to {SAVE_DIR}: table1_datasets.csv, table2_fold_matched_gaps.csv, table2b_mean_gap_by_pair_type.csv;"
      f" figures in {FIG_DIR}")


---
## Part 6 — Does putting craving features on a common time scale restore transfer? (added 17 Oct 2026)

Part 3 found that craving features computed **across entries** (last 1–3 ratings, 3-entry mean/SD, change since the
previous entry) transfer between the two prompted studies but not to or from Smoke Free, and we proposed that this is
because "the last three entries" spans a few hours in a prompted study but can span days in the app.

**Test.** Recompute the same information over fixed **clock-time** windows, identically in every dataset:

| Feature | Definition (previous entries only, except where stated) |
|---|---|
| `craving` | current craving (unchanged) |
| `crav_mean_6h`, `crav_sd_6h` | mean / SD of craving in the past 6 h (SD includes the current rating) |
| `crav_mean_24h`, `crav_sd_24h` | mean / SD of craving in the past 24 h (SD includes the current rating) |
| `crav_dev_24h` | current craving minus the past-24 h mean |
| `crav_ewm_6h` | time-decayed mean of earlier cravings, weight exp(−Δt / 6 h) |

If no earlier entry falls in a window, the mean is set to the current craving and the SD to 0 (no information about change),
so no feature depends on how many entries a design produces. Counts of entries per window are deliberately **not** used.

Two sets, same pairs and evaluation as Parts 3–4 (fold-matched gaps):
- `craving_time` (7) — the time-scaled counterpart of `craving_only` (7);
- `core_time` (17) — lapse history (4) + `craving_time` (7) + timing (6), the counterpart of `core`.

**Reading the result.** If the gap for pairs involving Smoke Free shrinks from about −0.16 (`craving_only`) towards 0 with
`craving_time`, the time-scale mechanism is supported. If it does not, the difference lies in what craving reports mean
under each design (selection on the predictor), not in the time window.

LR and XGBoost (about 20–30 minutes). Run Cells 1–4, then 14 and 15. Saved per configuration as before.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 14 — PART 6: TIME-SCALED CRAVING FEATURES AND TRANSFER RUNS
# ════════════════════════════════════════════════════════
def add_time_scaled(d, windows=(6, 24), tau=6.0):
    d = d.sort_values(['id', 't']).reset_index(drop=True)
    out = {f'crav_mean_{w}h': np.zeros(len(d)) for w in windows}
    out.update({f'crav_sd_{w}h': np.zeros(len(d)) for w in windows})
    out['crav_ewm_6h'] = np.zeros(len(d))
    for _, idx in d.groupby('id', sort=False).indices.items():
        idx = np.sort(idx)
        th = (d['t'].iloc[idx] - EPOCH).dt.total_seconds().to_numpy() / 3600.0
        c = d['craving'].iloc[idx].to_numpy(float)
        cs = np.concatenate([[0.], np.cumsum(c)]); cs2 = np.concatenate([[0.], np.cumsum(c ** 2)])
        pos = np.arange(len(c))
        for w in windows:
            left = np.searchsorted(th, th - w, side='left')
            n_prev = pos - left                                     # earlier entries inside the window
            mean_prev = np.where(n_prev > 0, (cs[pos] - cs[left]) / np.maximum(n_prev, 1), c)
            n_inc = n_prev + 1                                       # including the current rating
            s1 = cs[pos + 1] - cs[left]; s2 = cs2[pos + 1] - cs2[left]
            var = np.where(n_inc > 1, (s2 - s1 ** 2 / n_inc) / np.maximum(n_inc - 1, 1), 0.)
            out[f'crav_mean_{w}h'][idx] = mean_prev
            out[f'crav_sd_{w}h'][idx] = np.sqrt(np.clip(var, 0, None))
        dt = th[:, None] - th[None, :]                               # hours between entry i and earlier entry j
        W = np.where(dt > 0, np.exp(-dt / tau), 0.) * np.tri(len(c), k=-1)
        sw = W.sum(1)
        out['crav_ewm_6h'][idx] = np.where(sw > 1e-9, (W @ c) / np.maximum(sw, 1e-12), c)
    for k, v in out.items(): d[k] = v
    d['crav_dev_24h'] = d['craving'] - d['crav_mean_24h']
    return d


for k in DATA: DATA[k] = add_time_scaled(DATA[k])
HISTORY  = ['prior_lapse', 'lapse_lag2', 'lapse_lag3', 'days_since_last_lapse']
TIMING   = ['mins_since_prev', 'days_since_start', 'log_days_since_qd', 'hour_sin', 'hour_cos', 'is_weekend']
CRAVING  = ['craving', 'severity_change', 'rolling3_sev_mean', 'rolling3_sev_std', 'sev_lag1', 'sev_lag2', 'sev_lag3']
CRAV_TIME = ['craving', 'crav_mean_6h', 'crav_sd_6h', 'crav_mean_24h', 'crav_sd_24h', 'crav_dev_24h', 'crav_ewm_6h']
FSETS['craving_only'] = CRAVING
FSETS['craving_time'] = CRAV_TIME
FSETS['core_time']    = HISTORY + CRAV_TIME + TIMING
for k, d in DATA.items():
    assert not d[CRAV_TIME].isna().any().any(), f"{k}: missing values in time-scaled features"

# how many earlier entries fall in each window (descriptive; shows why entry-based windows differ by design)
print("Median number of earlier entries within the window, per dataset:")
for k, d in DATA.items():
    g = d.groupby('id')
    th = (d['t'] - EPOCH).dt.total_seconds() / 3600
    res = {}
    for w in (6, 24):
        cnt = []
        for _, idx in g.indices.items():
            x = np.sort(th.iloc[idx].to_numpy()); cnt.append(np.arange(len(x)) - np.searchsorted(x, x - w, 'left'))
        res[f'{w}h'] = float(np.median(np.concatenate(cnt)))
    span3 = g['t'].transform(lambda s: (s - s.shift(3)).dt.total_seconds() / 3600).median()
    print(f"  {k}: {res}   hours spanned by the last 3 entries (median): {span3:.1f}")

P6_SETS, P6_MODELS = ['craving_time', 'core_time'], ['LR', 'XGB']


def fit_any(m, seqs, nf):
    return train_gru(seqs, nf, EPOCHS) if m == 'GRU' else fit_flat(m, seqs)


def predict_any(m, model, seqs):
    return predict_gru(model, seqs) if m == 'GRU' else predict_flat(model, seqs)


PLAN6 = [(s, t, fset, 'within' if s == t else 'per_dataset', m)
         for fset in P6_SETS for m in P6_MODELS for s in DATA for t in DATA]
todo6 = [c for c in PLAN6 if not done(f"E_{c[0]}_to_{c[1]}__{c[2]}__{c[3]}__{c[4]}")]
print(f"\n{len(PLAN6)} configurations, {len(PLAN6) - len(todo6)} already saved, {len(todo6)} to run -> {SAVE_DIR}")
t0 = time.time(); cache6 = {}
for (s, t, fset, sc, m) in todo6:
    name = f"E_{s}_to_{t}__{fset}__{sc}__{m}"
    feats = FSETS[fset]; nf = len(feats); set_seed(SEED)
    if s == t:
        allseq = seqs_of(DATA[s], feats); rows = []
        for k, (tr, te) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(allseq)):
            set_seed(SEED + k)
            train = [allseq[i] for i in tr]; test = [allseq[i] for i in te]
            mu, sd = stats_of(train)
            model = fit_any(m, standardise(train, mu, sd), nf)
            rows += to_rows(predict_any(m, model, standardise(test, mu, sd)), fold=k)
    else:
        key = (s, fset, m)
        src = seqs_of(DATA[s], feats); mu_s, sd_s = stats_of(src)
        if key not in cache6: cache6[key] = fit_any(m, standardise(src, mu_s, sd_s), nf)
        tgt = seqs_of(DATA[t], feats); mu_t, sd_t = stats_of(tgt)
        rows = to_rows(predict_any(m, cache6[key], standardise(tgt, mu_t, sd_t)))
    tmp = pred_path(name) + '.tmp'
    pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))
    print(f"  {name:48s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 6 runs done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 15 — PART 6 RESULTS: ENTRY-BASED vs TIME-SCALED CRAVING FEATURES (fold-matched)
# For each pair: gap (transfer − within) with entry-based features, gap with time-scaled features, and the
# per-fold change in the gap (time − entry) with a 95% t-interval over the target's folds.
# ════════════════════════════════════════════════════════
def fold_aucs(d, col='p'):
    return pd.Series({k: auc(g['y'], g[col]) for k, g in d.groupby('fold')})


def t_ci(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return (x.mean() - h, x.mean() + h)


def fold_gaps(s, t, fset, m):
    a, w = f"E_{s}_to_{t}__{fset}__per_dataset__{m}", f"E_{t}_to_{t}__{fset}__within__{m}"
    if not (done(a) and done(w)): return None
    W_ = load_pred(w); A_ = load_pred(a); A_['fold'] = A_['id'].map(W_.groupby('id')['fold'].first())
    at, aw = fold_aucs(A_), fold_aucs(W_)
    pa, pw = per_person(A_), per_person(W_); common = pa.index.intersection(pw.index)
    return (at - aw), at.mean(), aw.mean(), float(np.median(pa[common] - pw[common])) if len(common) else np.nan


COMPARE = [('craving_only', 'craving_time'), ('core', 'core_time')]
rows = []
for m in P6_MODELS:
    for s in DATA:
        for t in DATA:
            if s == t: continue
            for old, new in COMPARE:
                go, gn = fold_gaps(s, t, old, m), fold_gaps(s, t, new, m)
                if go is None or gn is None: continue
                dk = (gn[0] - go[0]).dropna(); lo, hi = t_ci(dk)
                rows.append(dict(model=m, comparison=f"{old} → {new}", source=s, target=t,
                                 pair_type='Prompted ↔ prompted' if 'SF' not in (s, t) else 'Involves Smoke Free',
                                 entry_transfer=round(go[1], 3), entry_within=round(go[2], 3),
                                 entry_gap=round(go[0].mean(), 3),
                                 time_transfer=round(gn[1], 3), time_within=round(gn[2], 3),
                                 time_gap=round(gn[0].mean(), 3),
                                 gap_change=round(dk.mean(), 3), CI=f"[{lo:+.3f}, {hi:+.3f}]",
                                 folds_improved=f"{int((dk > 0).sum())}/{len(dk)}",
                                 pp_gap_entry=round(go[3], 3), pp_gap_time=round(gn[3], 3)))
P6 = pd.DataFrame(rows)
P6.to_csv(os.path.join(SAVE_DIR, 'part6_time_scaled_gaps.csv'), index=False)
pd.set_option('display.width', 250)
for m in P6_MODELS:
    for cmp_ in dict.fromkeys(P6.comparison):
        sub = P6[(P6.model == m) & (P6.comparison == cmp_)]
        if sub.empty: continue
        print(f"\n=== {m}: {cmp_} — fold-matched gap (transfer − within); gap_change > 0 = time-scaling helps transfer ===")
        print(sub.drop(columns=['model', 'comparison']).to_string(index=False))

print("\n=== Mean fold-matched gap by pair type ===")
print(P6.pivot_table(index=['model', 'comparison', 'pair_type'], values=['entry_gap', 'time_gap', 'gap_change',
                                                                            'entry_within', 'time_within']).round(3).to_string())

# within-study discrimination: does time-scaling cost anything inside each study?
print("\n=== Within-study fold-averaged AUC (entry-based vs time-scaled) ===")
wrows = []
for m in P6_MODELS:
    for t in DATA:
        r = {'model': m, 'dataset': t}
        for fset in ['craving_only', 'craving_time', 'core', 'core_time']:
            nm = f"E_{t}_to_{t}__{fset}__within__{m}"
            if done(nm): r[fset] = round(fold_aucs(load_pred(nm)).mean(), 3)
        wrows.append(r)
print(pd.DataFrame(wrows).to_string(index=False))

# direction of each time-scaled feature, per dataset (univariable standardised LR; pooled and within person)
coef = {}
for name, d0 in DATA.items():
    for f in CRAV_TIME:
        for kind in ['pooled', 'within']:
            d1 = d0.copy()
            if kind == 'within': d1[f] = d1[f] - d1.groupby('id')[f].transform('mean')
            sq = seqs_of(d1, [f]); mu, sd = stats_of(sq)
            coef[(f, kind, name)] = round(float(fit_flat('LR', standardise(sq, mu, sd)).coef_[0][0]), 3)
COEF6 = pd.Series(coef).rename_axis(['feature', 'kind', 'dataset']).unstack('dataset')[list(DATA)]
COEF6['same_sign_all'] = np.sign(COEF6[list(DATA)]).nunique(axis=1) == 1
COEF6.to_csv(os.path.join(SAVE_DIR, 'part6_feature_directions.csv'))
print("\n=== Direction of each time-scaled feature (univariable standardised LR coefficient) ===")
print(COEF6.to_string())
print(f"\nSaved: part6_time_scaled_gaps.csv, part6_feature_directions.csv in {SAVE_DIR}")


---
## Part 7 — Does a within-person ranking model transfer across studies? (added 17 Oct 2026)

Parts 1–6 used models trained with ordinary pooled loss. Within-person prediction was best with target-only training.
Here we test the model from Modiridovom, Akopian & Chalela (*From Population Discrimination to Personalized Risk*,
under review), which is trained specifically for within-person ranking. The implementation follows the authors' code
(`02_trend_ablation_and_ranking_loss.py`):

- **Features (common to all three studies):** craving, days since start, minutes since previous entry,
  **self-referential** craving (z-score and expanding mean of the person's own craving), and **aggregate** lapse history
  (lapses in the last 5 entries, cumulative lapse rate). Next-entry task (features at *t* → lapse at *t* + 1).
- **Model:** 1-layer GRU (32 units); self-supervised masked-feature reconstruction (15% masked, 40 epochs, lr 5e-3),
  then 40 epochs of joint training (lr 2.5e-3) with loss = 0.3 × BCE + 1.0 × within-person pairwise ranking loss;
  full-batch training.
- **Sample:** participants with both lapse and non-lapse entries and ≥ 6 entries (as in the original code).

**Conditions**
| Name | Features | Loss |
|---|---|---|
| `RANK` (final model) | self-ref + aggregate | 0.3 BCE + 1.0 ranking |
| `RANK_noagg` | self-ref only | 0.3 BCE + 1.0 ranking |
| `BCE` (control) | self-ref + aggregate | BCE only |

**Experiments**
- **7A Direct transfer:** train on all eligible source participants (3 seeds), predict every eligible target participant;
  compare per participant with the same model trained within the target (5-fold CV × 3 repeats), all six pairs.
- **7B Fine-tuning:** for EMASENS and the alcohol study, train on *n* target participants (outer 5-fold CV, 3 repeats):
  target only vs. fine-tuning a `RANK` model pretrained on Smoke Free vs. on the other prompted study, plus each
  pretrained model with no target data.

**Main metric:** per-participant AUC (median), paired by participant; pooled AUC reported alongside.
Run Cells 1–4, then 16 and 17. Each configuration is saved as it finishes.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 16 — PART 7: WITHIN-PERSON RANKING MODEL (Modiridovom et al.) — TRANSFER AND FINE-TUNING
# ════════════════════════════════════════════════════════
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

R_HIDDEN, R_SSL_EPOCHS, R_JOINT_EPOCHS, R_LR = 32, 40, 40, 5e-3
R_BCE_W, R_RANK_W, R_MASK = 0.3, 1.0, 0.15
R_FOLDS, R_REPEATS, R_SEEDS = 5, 3, 3
R_FT_EPOCHS, R_FT_LR = 20, 1e-3
R_N = {'EM': [5, 10, 15], 'AL': [10, 20, 40, 70]}
if QUICK_TEST:
    R_SSL_EPOCHS = R_JOINT_EPOCHS = R_FT_EPOCHS = 2; R_REPEATS = R_SEEDS = 1; R_FOLDS = 2
    R_N = {'EM': [3], 'AL': [5]}
R_BASE = ['craving', 'days_since_start', 'mins_since_prev']
R_SELF = ['sev_z', 'sev_base']
R_AGG  = ['lapse_last5', 'cum_rate']
R_COND = {'RANK': (R_BASE + R_SELF + R_AGG, True), 'RANK_noagg': (R_BASE + R_SELF, True),
          'BCE': (R_BASE + R_SELF + R_AGG, False)}


def add_rank_features(d):
    d = d.sort_values(['id', 't']).reset_index(drop=True); g = d.groupby('id', sort=False)
    m = g['craving'].transform(lambda v: v.expanding().mean())
    s = g['craving'].transform(lambda v: v.expanding().std()).fillna(0).replace(0, 1e-6)
    d['sev_z'] = (d['craving'] - m) / s; d['sev_base'] = m
    d['lapse_last5'] = g['label'].transform(lambda v: v.rolling(5, min_periods=1).sum())
    d['cum_rate'] = g['label'].transform(lambda v: v.expanding().mean())
    return d


RD = {}
for k, d in DATA.items():
    d = add_rank_features(d)
    n = d.groupby('id').size(); lr_ = d.groupby('id')['label'].mean()
    keep = n[(n >= 6)].index.intersection(lr_[(lr_ > 0) & (lr_ < 1)].index)
    d = d[d['id'].isin(keep)].reset_index(drop=True)
    feats = R_BASE + R_SELF + R_AGG                          # standardise each dataset with its own feature statistics
    mu, sd = d[feats].mean(), d[feats].std().replace(0, 1)
    d[feats] = (d[feats] - mu) / sd
    RD[k] = d
    print(f"{k}: {d['id'].nunique()} eligible participants (both outcomes, ≥6 entries), {len(d):,} entries")


def r_seqs(d, feats):
    return {pid: (torch.tensor(g[feats].to_numpy(np.float32)[:-1]), torch.tensor(g['label'].to_numpy(np.float32)[1:]),
                  g['cum_rate'].to_numpy()[:-1]) for pid, g in d.groupby('id', sort=True) if len(g) >= 2}


class RankGRU(nn.Module):
    def __init__(self, nf, h=R_HIDDEN):
        super().__init__()
        self.gru = nn.GRU(nf, h, batch_first=True); self.decoder = nn.Linear(h, nf); self.cls_head = nn.Linear(h, 1)
    def encode(self, x, L):
        pk = pack_padded_sequence(x, L, batch_first=True, enforce_sorted=False)
        return pad_packed_sequence(self.gru(pk)[0], batch_first=True, total_length=x.size(1))[0]
    def reconstruct(self, x, L): return self.decoder(self.encode(x, L))
    def classify(self, x, L): return self.cls_head(self.encode(x, L)).squeeze(-1)


def r_batch(seqs, pids):
    X = pad_sequence([seqs[p][0] for p in pids], batch_first=True)
    Y = pad_sequence([seqs[p][1] for p in pids], batch_first=True)
    L = [len(seqs[p][0]) for p in pids]
    M = torch.zeros(len(L), X.size(1))
    for i, l in enumerate(L): M[i, :l] = 1
    return X.to(device), Y.to(device), L, M.to(device)


def rank_loss(logits, Y, M):
    tot, n = 0., 0
    for i in range(logits.size(0)):
        v = M[i].bool(); y, s_ = Y[i][v], logits[i][v]
        pos, neg = s_[y == 1], s_[y == 0]
        if pos.numel() == 0 or neg.numel() == 0: continue
        pl = F.softplus(-(pos.unsqueeze(1) - neg.unsqueeze(0))); tot = tot + pl.sum(); n += pl.numel()
    return tot / n if n else torch.tensor(0., device=logits.device, requires_grad=True)


def r_train(seqs, pids, nf, use_rank, seed, init=None, ssl=True, epochs=None, lr=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = RankGRU(nf).to(device)
    if init is not None: model.load_state_dict(init)
    X, Y, L, M = r_batch(seqs, pids)
    if ssl:
        opt = torch.optim.Adam(model.parameters(), lr=R_LR)
        for _ in range(R_SSL_EPOCHS):
            opt.zero_grad(); fm = (torch.rand_like(X) < R_MASK).float()
            loss = ((F.mse_loss(model.reconstruct(X * (1 - fm), L), X, reduction='none').mean(-1) * M).sum() / M.sum())
            loss.backward(); opt.step()
    opt = torch.optim.Adam(model.parameters(), lr=lr or R_LR * 0.5)
    for _ in range(epochs or R_JOINT_EPOCHS):
        opt.zero_grad(); lg = model.classify(X, L)
        bce = (F.binary_cross_entropy_with_logits(lg, Y, reduction='none') * M).sum() / M.sum()
        loss = (R_BCE_W * bce + R_RANK_W * rank_loss(lg, Y, M)) if use_rank else bce
        loss.backward(); opt.step()
    return model


def r_predict(model, seqs, pids, **tags):
    model.eval(); rows = []
    with torch.no_grad():
        for p in pids:
            x, y, base = seqs[p]
            pr = torch.sigmoid(model.classify(x.unsqueeze(0).to(device), [len(x)]))[0].cpu().numpy()
            rows += [dict(tags, id=p, y=int(a), p=float(b), base=float(c)) for a, b, c in zip(y.numpy(), pr, base)]
    return rows


def r_save(name, rows):
    tmp = pred_path(name) + '.tmp'; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))


t0 = time.time()
# ── 7A: within-target reference (5-fold × repeats) and direct transfer (all source participants × seeds) ──
for cond, (feats, use_rank) in R_COND.items():
    S = {k: r_seqs(RD[k], feats) for k in RD}
    for t in RD:
        name = f"R7_{t}_to_{t}__{cond}"
        if not done(name):
            ids = np.array(sorted(S[t])); rows = []
            for r in range(R_REPEATS):
                perm = np.random.default_rng(SEED + r).permutation(ids)
                for k, te in enumerate(np.array_split(perm, R_FOLDS)):
                    tr = [p for p in ids if p not in set(te)]
                    mdl = r_train(S[t], tr, len(feats), use_rank, seed=SEED + 100 * r + k)
                    rows += r_predict(mdl, S[t], list(te), repeat=r, fold=k)
            r_save(name, rows); print(f"  {name:32s} {(time.time()-t0)/60:6.1f} min")
    for s in RD:
        todo = [t for t in RD if t != s and not done(f"R7_{s}_to_{t}__{cond}")]
        if not todo: continue
        models = [r_train(S[s], sorted(S[s]), len(feats), use_rank, seed=SEED + j) for j in range(R_SEEDS)]
        for t in todo:
            rows = []
            for j, mdl in enumerate(models): rows += r_predict(mdl, S[t], sorted(S[t]), repeat=j)
            r_save(f"R7_{s}_to_{t}__{cond}", rows); print(f"  R7_{s}_to_{t}__{cond:24s} {(time.time()-t0)/60:6.1f} min")

# ── 7B: fine-tuning curves for the prompted studies ──
feats, _ = R_COND['RANK']; S = {k: r_seqs(RD[k], feats) for k in RD}
pre_path_ = lambda s: os.path.join(SAVE_DIR, f"rank_pretrained_{s}.pt")
for s in RD:
    if not os.path.exists(pre_path_(s)):
        torch.save(r_train(S[s], sorted(S[s]), len(feats), True, seed=SEED).state_dict(), pre_path_(s))
PRE = {s: torch.load(pre_path_(s), map_location=device) for s in RD}
for tgt in ['EM', 'AL']:
    other = 'AL' if tgt == 'EM' else 'EM'
    ids = np.array(sorted(S[tgt]))
    name0 = f"R7B_{tgt}__source_only"
    if not done(name0):
        rows = []
        for src in ['SF', other]:
            mdl = RankGRU(len(feats)).to(device); mdl.load_state_dict(PRE[src])
            rows += r_predict(mdl, S[tgt], list(ids), cond=f'source_only_{src}')
        r_save(name0, rows)
    for n in R_N[tgt]:
        for r in range(R_REPEATS):
            name = f"R7B_{tgt}__n{n:03d}__r{r}"
            if done(name): continue
            rows = []
            perm = np.random.default_rng(SEED + r).permutation(ids)
            for k, te in enumerate(np.array_split(perm, R_FOLDS)):
                tr = [p for p in ids if p not in set(te)]
                pick = list(np.random.default_rng(SEED + 100 * r + k).choice(tr, min(n, len(tr)), replace=False))
                sd_ = SEED + 100 * r + k
                rows += r_predict(r_train(S[tgt], pick, len(feats), True, seed=sd_), S[tgt], list(te),
                                  cond='target_only', fold=k)
                for src in ['SF', other]:
                    ft = r_train(S[tgt], pick, len(feats), True, seed=sd_, init=PRE[src], ssl=False,
                                 epochs=R_FT_EPOCHS, lr=R_FT_LR)
                    rows += r_predict(ft, S[tgt], list(te), cond=f'finetune_{src}', fold=k)
            r_save(name, pd.DataFrame(rows).assign(n=n, repeat=r).to_dict('records'))
            print(f"  {name:32s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 7 runs done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 17 — PART 7 RESULTS
# ════════════════════════════════════════════════════════
def pp_auc(d, col='p'):
    """Per-participant AUC, averaged over repeats/seeds within participant."""
    by = ['id'] + (['repeat'] if 'repeat' in d else [])
    a = d.groupby(by).apply(lambda g: auc(g['y'], g[col])).dropna()
    return a.groupby(level=0).mean()


def pooled_mean(d, col='p'):
    if 'repeat' not in d: return auc(d['y'], d[col])
    return np.nanmean([auc(g['y'], g[col]) for _, g in d.groupby('repeat')])


# 7A
rows = []
for cond in R_COND:
    for t in RD:
        w = load_pred(f"R7_{t}_to_{t}__{cond}"); aw = pp_auc(w)
        rows.append(dict(cond=cond, source=t, target=t, kind='within', n_pp=len(aw),
                         pp_median=round(aw.median(), 3), pooled=round(pooled_mean(w), 3),
                         baseline_pp_median=round(pp_auc(w, 'base').median(), 3)))
        for s in RD:
            if s == t or not done(f"R7_{s}_to_{t}__{cond}"): continue
            a = pp_auc(load_pred(f"R7_{s}_to_{t}__{cond}")); c = a.index.intersection(aw.index)
            dlt = (a[c] - aw[c]).to_numpy()
            rows.append(dict(cond=cond, source=s, target=t, kind='transfer', n_pp=len(c),
                             pp_median=round(a[c].median(), 3), pooled=round(pooled_mean(load_pred(f"R7_{s}_to_{t}__{cond}")), 3),
                             within_pp_median=round(aw[c].median(), 3),
                             pp_diff_median=round(float(np.median(dlt)), 3), share_transfer_better=round(float((dlt > 0).mean()), 2),
                             wilcoxon_p=stats.wilcoxon(dlt).pvalue if np.any(dlt != 0) else 1.0,
                             pair_type='Prompted ↔ prompted' if 'SF' not in (s, t) else 'Involves Smoke Free'))
R7A = pd.DataFrame(rows)
R7A.to_csv(os.path.join(SAVE_DIR, 'part7A_rank_transfer.csv'), index=False)
pd.set_option('display.width', 250)
print("=== 7A within-target reference (5-fold CV) ===")
print(R7A[R7A.kind == 'within'][['cond', 'target', 'n_pp', 'pp_median', 'pooled', 'baseline_pp_median']].to_string(index=False))
print("\n=== 7A direct transfer vs within-target (per-participant AUC, paired) ===")
print(R7A[R7A.kind == 'transfer'][['cond', 'source', 'target', 'pair_type', 'n_pp', 'pp_median', 'within_pp_median',
                                   'pp_diff_median', 'share_transfer_better', 'wilcoxon_p', 'pooled']]
      .round(4).to_string(index=False))

# does the ranking objective transfer better than BCE? (paired per participant, transfer AUCs)
cmp_rows = []
for s in RD:
    for t in RD:
        if s == t: continue
        A = {c: pp_auc(load_pred(f"R7_{s}_to_{t}__{c}")) for c in R_COND}
        for x, y in [('RANK', 'BCE'), ('RANK', 'RANK_noagg')]:
            c = A[x].index.intersection(A[y].index); dlt = (A[x][c] - A[y][c]).to_numpy()
            cmp_rows.append(dict(source=s, target=t, comparison=f"{x} − {y}", n_pp=len(c),
                                 median_diff=round(float(np.median(dlt)), 3),
                                 wilcoxon_p=round(stats.wilcoxon(dlt).pvalue, 4) if np.any(dlt != 0) else 1.0))
print("\n=== Transferred models: ranking vs BCE objective, and with vs without aggregate features (per participant) ===")
print(pd.DataFrame(cmp_rows).to_string(index=False))

# 7B
e = []
for tgt in ['EM', 'AL']:
    d0 = load_pred(f"R7B_{tgt}__source_only")
    for f in sorted(glob.glob(pred_path(f"R7B_{tgt}__n*"))):
        d = pd.read_csv(f, dtype={'id': str}); n, r = int(d['n'].iloc[0]), int(d['repeat'].iloc[0])
        for cond, g in d.groupby('cond'):
            a = g.groupby('id').apply(lambda h: auc(h['y'], h['p'])).dropna()
            e.append(dict(target=tgt, n=n, repeat=r, cond=cond, pp_median=a.median(), pooled=auc(g['y'], g['p'])))
    for cond, g in d0.groupby('cond'):
        a = g.groupby('id').apply(lambda h: auc(h['y'], h['p'])).dropna()
        e.append(dict(target=tgt, n=0, repeat=0, cond=cond, pp_median=a.median(), pooled=auc(g['y'], g['p'])))
R7B = pd.DataFrame(e)
R7Bs = R7B.groupby(['target', 'cond', 'n'])[['pp_median', 'pooled']].agg(['mean', 'std']).round(3)
R7Bs.to_csv(os.path.join(SAVE_DIR, 'part7B_rank_finetune.csv'))
print("\n=== 7B fine-tuning curves (per-participant median AUC and pooled AUC; mean ± SD over repeats) ===")
print(R7Bs.to_string())
print(f"\nSaved: part7A_rank_transfer.csv, part7B_rank_finetune.csv in {SAVE_DIR}")


---
## Part 8 — One model trained on all three studies (added 18 Oct 2026)

Instead of transferring a model from one study to another, train **one model on all three studies** and test it on each
study separately (participant-level cross-validation in the target study; the other studies are always in training).

| Name | Training data | Features |
|---|---|---|
| `A_core` (reference) | target study only | 17 common-core features |
| `A_all` (reference) | target study only | all features available in the target study |
| `B_pool3` | all three studies | common core |
| `B_pool3_bal` | all three studies, **each study weighted equally** | common core |
| `C_pool3_ind` | all three studies | common core + study indicator |
| `D0_pool3_zero` | all three studies | **all features** (61); missing features filled with 0 |
| `D_pool3_miss` | all three studies | all features; missing = 0 **plus missing-feature indicators** |
| `E_pool2` / `E_pool2_bal` | the two **prompted** studies only (targets EMASENS, alcohol) | common core |

- **All features:** the 17 core features; 20 smoking items shared by Smoke Free and EMASENS (5 feelings plus
  stressed, 7 activities, driving, 4 social contexts, cigarettes per day, time to first cigarette); 10 Smoke Free-only
  items; 9 EMASENS-only items; 5 alcohol-only items. Each feature is standardised within the study where it exists;
  absent features are 0. The missing indicators necessarily identify the study (structured missingness), so D answers
  "can one deployed model serve every study?", not "do the features generalise?".
- **Folds:** the same 10 participant folds as Parts 1–4, so every comparison with `A` is fold-matched.
- **Models:** logistic regression and XGBoost (Cell 18), and the within-person ranking GRU from Part 7 (Cell 18b;
  pooled over all three studies vs the two prompted studies, same folds and repeats as Part 7).
- **Metrics:** fold-matched pooled AUC (difference from `A_core`, 95% t-interval over folds) and per-participant AUC
  (median difference from `A_core`, Wilcoxon).

Run Cells 1–4, then 18, 18b and 19. Everything is saved as it finishes.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 18 — PART 8: POOLED TRAINING (LR, XGBoost)
# ════════════════════════════════════════════════════════
SMOKE_SHARED = ['anxious', 'bored', 'happy', 'annoyed', 'down', 'stressed', 'act_working', 'act_reading', 'act_relaxing',
                'act_eating', 'act_social', 'caffeine', 'alcohol', 'driving', 'soc_alone', 'soc_partner', 'soc_friends',
                'soc_family', 'cpd', 'ttfc']
SF_ONLY, EM_ONLY, AL_ONLY = list(SF_EXTRA_COLS), list(EM_EXTRA_COLS), list(AL_EXTRA_COLS)
ALL_FEATS = CORE + SMOKE_SHARED + SF_ONLY + EM_ONLY + AL_ONLY
PRESENT = {'SF': CORE + SMOKE_SHARED + SF_ONLY, 'EM': CORE + SMOKE_SHARED + EM_ONLY, 'AL': CORE + AL_ONLY}
GROUPS = {'smoke': SMOKE_SHARED, 'sf': SF_ONLY, 'em': EM_ONLY, 'al': AL_ONLY}
print(f"All features: {len(ALL_FEATS)} ({len(CORE)} core + {len(SMOKE_SHARED)} smoking-shared + "
      f"{len(SF_ONLY)} SF-only + {len(EM_ONLY)} EM-only + {len(AL_ONLY)} AL-only)")
for k, d in DATA.items():
    assert not d[PRESENT[k]].isna().any().any(), f"{k}: missing values in {d[PRESENT[k]].columns[d[PRESENT[k]].isna().any()].tolist()}"


def build(k, ids, feats_mode, mu_sd=None):
    """Sequences for study k (participants `ids`, or all), standardised within study, in the requested feature layout.
    Returns seqs [(X, y, id, run_rate)] and the (mu, sd) used."""
    present = CORE if feats_mode in ('core', 'core_ind') else PRESENT[k]
    sq = seqs_of(DATA[k], present, ids)
    mu, sd = mu_sd if mu_sd is not None else stats_of(sq)
    sq = standardise(sq, mu, sd)
    if feats_mode == 'core': return sq, (mu, sd)
    out = []
    for X, y, pid, rr in sq:
        n = len(X)
        if feats_mode == 'core_ind':
            Z = np.hstack([X, np.tile([[k == 'SF', k == 'EM']], (n, 1)).astype(np.float32)])
        else:
            Z = np.zeros((n, len(ALL_FEATS)), np.float32)
            Z[:, [ALL_FEATS.index(f) for f in present]] = X
            if feats_mode == 'all_miss':
                miss = np.array([[not set(g) <= set(present) for g in GROUPS.values()]], np.float32)
                Z = np.hstack([Z, np.tile(miss, (n, 1))])
        out.append((Z, y, pid, rr))
    return out, (mu, sd)


def fit_flat_w(mtype, seqs, w_seq):
    X, y = flat_next(seqs); y = y.astype(int)
    w = np.concatenate([np.full(len(s[1]) - 1, wi) for s, wi in zip(seqs, w_seq)]); w = w / w.mean()
    pos = max((y == 1).sum(), 1); neg = (y == 0).sum()
    if mtype == 'LR':
        clf = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', random_state=42)
    else:
        clf = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                                scale_pos_weight=neg / pos, eval_metric='logloss', n_jobs=-1, random_state=42, verbosity=0)
    clf.fit(X, y, sample_weight=w)
    return clf


P8_COND = {   # name: (training studies ('all' | 'prompted' | 'self'), feature layout, balanced weights)
    'A_all': ('self', 'all_self', False),
    'B_pool3': ('all', 'core', False), 'B_pool3_bal': ('all', 'core', True),
    'C_pool3_ind': ('all', 'core_ind', False),
    'D0_pool3_zero': ('all', 'all_zero', False), 'D_pool3_miss': ('all', 'all_miss', False),
    'E_pool2': ('prompted', 'core', False), 'E_pool2_bal': ('prompted', 'core', True)}
P8_MODELS = ['LR', 'XGB']

t0 = time.time()
for m in P8_MODELS:
    for t in DATA:
        ref = f"E_{t}_to_{t}__core__within__{m}"                 # A_core: same as Parts 1–4 (rerun only if missing)
        all_ids = [s_[2] for s_ in seqs_of(DATA[t], CORE)]
        folds = list(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(all_ids))
        conds = ([('A_core', None)] if not done(ref) else []) + list(P8_COND.items())
        for cname, spec in conds:
            if spec is not None and spec[0] == 'prompted' and t == 'SF': continue
            name = ref if cname == 'A_core' else f"P8_{t}__{cname}__{m}"
            if done(name): continue
            src, layout, bal = spec if spec is not None else ('self', 'core', False)
            if layout == 'all_self': layout = 'all_zero'
            others = [] if src == 'self' else [k for k in DATA if k != t and (src == 'all' or k != 'SF')]
            other_seqs = {k: build(k, None, layout)[0] for k in others}
            rows = []
            for k_, (tr, te) in enumerate(folds):
                set_seed(SEED + k_)
                tr_ids = [all_ids[i] for i in tr]; te_ids = [all_ids[i] for i in te]
                tr_s, musd = build(t, tr_ids, layout); te_s, _ = build(t, te_ids, layout, musd)
                train = tr_s + [x for k in others for x in other_seqs[k]]
                sizes = {t: sum(len(x[1]) for x in tr_s), **{k: sum(len(x[1]) for x in other_seqs[k]) for k in others}}
                w_seq = ([1.0 / sizes[t]] * len(tr_s) + [1.0 / sizes[k] for k in others for _ in other_seqs[k]]) if bal \
                    else [1.0] * len(train)
                if layout == 'all_zero' and src == 'self':          # drop all-zero columns for a single study
                    keep = [ALL_FEATS.index(f) for f in PRESENT[t]]
                    train = [(X[:, keep], y, p_, r_) for X, y, p_, r_ in train]; te_s = [(X[:, keep], y, p_, r_) for X, y, p_, r_ in te_s]
                clf = fit_flat_w(m, train, w_seq)
                rows += to_rows(predict_flat(clf, te_s), fold=k_)
            tmp = pred_path(name) + '.tmp'; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))
            print(f"  {name:36s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 8 (LR, XGBoost) done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 18b — PART 8: POOLED TRAINING OF THE WITHIN-PERSON RANKING GRU (Part 7 model and folds)
# Needs Cell 16 to have been run in this session (RD, r_seqs, r_train, r_predict, R_COND).
# ════════════════════════════════════════════════════════
feats, use_rank = R_COND['RANK']
S8 = {k: r_seqs(RD[k], feats) for k in RD}
for t in RD:
    for cname, others in [('pool3', [k for k in RD if k != t]), ('pool2', [k for k in RD if k not in (t, 'SF')])]:
        if t == 'SF' and cname == 'pool2': continue
        name = f"R8_{t}__{cname}"
        if done(name): continue
        ids = np.array(sorted(S8[t])); rows = []
        pool = {}
        for k in others: pool.update({(k, p): v for p, v in S8[k].items()})
        for r in range(R_REPEATS):
            perm = np.random.default_rng(SEED + r).permutation(ids)          # identical folds to Part 7
            for k_, te in enumerate(np.array_split(perm, R_FOLDS)):
                tr = [p for p in ids if p not in set(te)]
                seqs = {**{('T', p): S8[t][p] for p in tr}, **pool}
                mdl = r_train(seqs, list(seqs), len(feats), True, seed=SEED + 100 * r + k_)
                rows += r_predict(mdl, S8[t], list(te), repeat=r, fold=k_)
        r_save(name, rows); print(f"  {name:24s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 8 (ranking GRU) done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 19 — PART 8 RESULTS
# ════════════════════════════════════════════════════════
def pp_auc(d, col='p'):                       # per-participant AUC, averaged over repeats (as in Cell 17)
    by = ['id'] + (['repeat'] if 'repeat' in d else [])
    a = d.groupby(by).apply(lambda g: auc(g['y'], g[col])).dropna()
    return a.groupby(level=0).mean()


def pooled_mean(d, col='p'):
    if 'repeat' not in d: return auc(d['y'], d[col])
    return np.nanmean([auc(g['y'], g[col]) for _, g in d.groupby('repeat')])


def fold_aucs(d, col='p'):
    return pd.Series({k: auc(g['y'], g[col]) for k, g in d.groupby('fold')})


def t_ci(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return (x.mean() - h, x.mean() + h)


rows = []
for m in P8_MODELS:
    for t in DATA:
        A = load_pred(f"E_{t}_to_{t}__core__within__{m}"); fa_A = fold_aucs(A); pp_A = per_person(A)
        rows.append(dict(model=m, target=t, condition='A_core', fa_auc=round(fa_A.mean(), 3), diff=0.0, CI='',
                         folds_better='', pp_median=round(pp_A.median(), 3), pp_diff=0.0, pp_p=np.nan,
                         baseline=round(fold_aucs(A, 'base').mean(), 3)))
        for c in P8_COND:
            nm = f"P8_{t}__{c}__{m}"
            if not done(nm): continue
            Bd = load_pred(nm); fa_B = fold_aucs(Bd); dk = (fa_B - fa_A).dropna(); lo, hi = t_ci(dk)
            pp_B = per_person(Bd); cm = pp_B.index.intersection(pp_A.index); dd = (pp_B[cm] - pp_A[cm]).to_numpy()
            rows.append(dict(model=m, target=t, condition=c, fa_auc=round(fa_B.mean(), 3), diff=round(dk.mean(), 3),
                             CI=f"[{lo:+.3f}, {hi:+.3f}]", folds_better=f"{int((dk > 0).sum())}/{len(dk)}",
                             pp_median=round(pp_B.median(), 3), pp_diff=round(float(np.median(dd)), 3),
                             pp_p=round(stats.wilcoxon(dd).pvalue, 4) if np.any(dd != 0) else 1.0, baseline=np.nan))
P8 = pd.DataFrame(rows)
P8.to_csv(os.path.join(SAVE_DIR, 'part8_pooled_training.csv'), index=False)
pd.set_option('display.width', 250)
print("=== Part 8: pooled training vs target-only (A_core). diff = fold-matched pooled AUC difference; "
      "pp = per-participant median AUC ===")
for m in P8_MODELS:
    print(f"\n--- {m} ---"); print(P8[P8.model == m].drop(columns='model').to_string(index=False))

# ranking GRU
rows = []
for t in RD:
    W = load_pred(f"R7_{t}_to_{t}__RANK"); aw = pp_auc(W)
    rows.append(dict(target=t, condition='within (Part 7)', pp_median=round(aw.median(), 3),
                     pooled=round(pooled_mean(W), 3), pp_diff=0.0, share_better=np.nan, p=np.nan))
    for c in ['pool3', 'pool2']:
        nm = f"R8_{t}__{c}"
        if not done(nm): continue
        Pd = load_pred(nm); ap = pp_auc(Pd); cm = ap.index.intersection(aw.index); dd = (ap[cm] - aw[cm]).to_numpy()
        rows.append(dict(target=t, condition=c, pp_median=round(ap[cm].median(), 3), pooled=round(pooled_mean(Pd), 3),
                         pp_diff=round(float(np.median(dd)), 3), share_better=round(float((dd > 0).mean()), 2),
                         p=round(stats.wilcoxon(dd).pvalue, 4) if np.any(dd != 0) else 1.0))
R8 = pd.DataFrame(rows)
R8.to_csv(os.path.join(SAVE_DIR, 'part8_pooled_ranking_gru.csv'), index=False)
print("\n=== Ranking GRU: pooled training vs within-study (per participant, paired) ===")
print(R8.to_string(index=False))
print(f"\nSaved: part8_pooled_training.csv, part8_pooled_ranking_gru.csv in {SAVE_DIR}")


---
## Part 9 — Share what transfers, keep local what doesn't (added 18 Oct 2026)

Parts 2–8 showed that lapse history and current craving transfer across studies, whereas craving dynamics and timing
are study-specific. Part 9 builds that into the model instead of choosing between "pool everything" and "pool nothing".

- **9A Partial pooling (logistic regression).** Feature augmentation (Daumé III, 2007): every feature has a *shared*
  coefficient; craving dynamics, timing and the intercept also get a *study-specific* coefficient. The L2 penalty pulls
  the study-specific coefficients toward zero, i.e. toward the shared value, so each study borrows strength where its
  own data are thin. The strength of that pull is chosen inside each training fold (20% of target training participants,
  by median within-person AUC). Pooled over all three studies (`aug3`) and over the two prompted studies (`aug2`).
- **9B Multi-task ranking GRU.** Your within-person ranking model with a **shared GRU encoder and one output head per
  study**, trained on all studies at once (`mt3`) or on the two prompted studies (`mt2`). Same loss, pretraining and folds
  as Part 7.
- **9C Stacking.** A small logistic combiner learns, from the target study's training folds, how to weight the
  target-only, transferred, pooled and partial-pooling logistic models.

**Primary outcome (fixed before running):** per-participant AUC versus target-only training (`A_core` for logistic
models, Part 7 within-study model for the GRU), paired Wilcoxon test; fold-matched pooled AUC reported alongside.
Same folds as Parts 7–8. Run Cells 1–4, 16, 18, then 20, 21, 22, 23.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 20 — PART 9A: PARTIAL POOLING BY FEATURE AUGMENTATION (logistic regression)
# ════════════════════════════════════════════════════════
SHARED_ONLY = ['prior_lapse', 'lapse_lag2', 'lapse_lag3', 'days_since_last_lapse', 'craving']      # transfer everywhere
LOCAL = [f for f in CORE if f not in SHARED_ONLY]                                               # dynamics + timing
STUDIES = ['SF', 'EM', 'AL']
AUG_SCALES = [0.25, 0.5, 1.0, 2.0]          # larger = weaker pull toward the shared coefficients
if QUICK_TEST: AUG_SCALES = [0.5, 1.0]


def augment(seqs, study, scale):
    """[shared copy of all CORE features | study-specific copies of LOCAL features + study intercept, for this study]."""
    li = [CORE.index(f) for f in LOCAL]; j = STUDIES.index(study); nl = len(LOCAL) + 1
    out = []
    for X, y, pid, rr in seqs:
        Z = np.zeros((len(X), len(CORE) + nl * len(STUDIES)), np.float32)
        Z[:, :len(CORE)] = X
        base = len(CORE) + j * nl
        Z[:, base:base + len(LOCAL)] = X[:, li] * scale
        Z[:, base + len(LOCAL)] = scale                                   # study-specific intercept
        out.append((Z, y, pid, rr))
    return out


def fit_aug(train_by_study, scale):
    seqs = [s for k, sq in train_by_study.items() for s in augment(sq, k, scale)]
    X, y = flat_next(seqs)
    return LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced', random_state=42).fit(X, y.astype(int))


def median_pp_auc(clf, seqs, study, scale):
    rows = to_rows(predict_flat(clf, augment(seqs, study, scale)))
    d = pd.DataFrame(rows); pp = per_person(d)
    return pp.median() if len(pp) >= 3 else auc(d['y'], d['p'])


t0 = time.time()
for t in DATA:
    all_ids = [s_[2] for s_ in seqs_of(DATA[t], CORE)]
    folds = list(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(all_ids))
    for cname, others in [('aug3', [k for k in DATA if k != t]), ('aug2', [k for k in DATA if k not in (t, 'SF')])]:
        if t == 'SF' and cname == 'aug2': continue
        name = f"P9_{t}__{cname}__LR"
        if done(name): continue
        other = {k: build(k, None, 'core')[0] for k in others}
        rows, chosen = [], []
        for k_, (tr, te) in enumerate(folds):
            tr_ids = [all_ids[i] for i in tr]; te_ids = [all_ids[i] for i in te]
            # inner split of the target's training participants to choose the scale
            rng = np.random.default_rng(SEED + k_); inner = rng.permutation(tr_ids)
            n_val = max(2, int(0.2 * len(inner))); val_ids, fit_ids = list(inner[:n_val]), list(inner[n_val:])
            fit_s, musd = build(t, fit_ids, 'core'); val_s, _ = build(t, val_ids, 'core', musd)
            scores = {}
            for a in AUG_SCALES:
                clf = fit_aug({t: fit_s, **other}, a); scores[a] = median_pp_auc(clf, val_s, t, a)
            best = max(scores, key=lambda a: (np.nan_to_num(scores[a], nan=-1), -a)); chosen.append(best)
            tr_s, musd = build(t, tr_ids, 'core'); te_s, _ = build(t, te_ids, 'core', musd)
            clf = fit_aug({t: tr_s, **other}, best)
            rows += to_rows(predict_flat(clf, augment(te_s, t, best)), fold=k_)
        tmp = pred_path(name) + '.tmp'; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))
        print(f"  {name:28s} scales chosen {pd.Series(chosen).value_counts().to_dict()}  {(time.time()-t0)/60:5.1f} min")
print("✅ Part 9A done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 21 — PART 9B: MULTI-TASK WITHIN-PERSON RANKING GRU (shared encoder, one head per study)
# Needs Cell 16 in this session (RD, r_seqs, r_batch, rank_loss, R_* settings, r_save).
# ════════════════════════════════════════════════════════
class MTRankGRU(nn.Module):
    def __init__(self, nf, n_heads, h=R_HIDDEN):
        super().__init__()
        self.gru = nn.GRU(nf, h, batch_first=True); self.decoder = nn.Linear(h, nf); self.heads = nn.Linear(h, n_heads)
    def encode(self, x, L):
        pk = pack_padded_sequence(x, L, batch_first=True, enforce_sorted=False)
        return pad_packed_sequence(self.gru(pk)[0], batch_first=True, total_length=x.size(1))[0]
    def reconstruct(self, x, L): return self.decoder(self.encode(x, L))
    def classify(self, x, L, head):                       # head: LongTensor (batch,) — study index of each sequence
        out = self.heads(self.encode(x, L))               # (batch, time, n_heads)
        return out.gather(2, head.view(-1, 1, 1).expand(-1, out.size(1), 1)).squeeze(-1)


def mt_train(seqs, keys, head_of, nf, n_heads, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MTRankGRU(nf, n_heads).to(device)
    X, Y, L, M = r_batch(seqs, keys)
    H = torch.tensor([head_of[k] for k in keys], device=device)
    opt = torch.optim.Adam(model.parameters(), lr=R_LR)
    for _ in range(R_SSL_EPOCHS):
        opt.zero_grad(); fm = (torch.rand_like(X) < R_MASK).float()
        loss = ((F.mse_loss(model.reconstruct(X * (1 - fm), L), X, reduction='none').mean(-1) * M).sum() / M.sum())
        loss.backward(); opt.step()
    opt = torch.optim.Adam(model.parameters(), lr=R_LR * 0.5)
    for _ in range(R_JOINT_EPOCHS):
        opt.zero_grad(); lg = model.classify(X, L, H)
        bce = (F.binary_cross_entropy_with_logits(lg, Y, reduction='none') * M).sum() / M.sum()
        (R_BCE_W * bce + R_RANK_W * rank_loss(lg, Y, M)).backward(); opt.step()
    return model


def mt_predict(model, seqs, pids, head, **tags):
    model.eval(); rows = []
    with torch.no_grad():
        for p in pids:
            x, y, base = seqs[p]
            lg = model.classify(x.unsqueeze(0).to(device), [len(x)], torch.tensor([head], device=device))
            pr = torch.sigmoid(lg)[0].cpu().numpy()
            rows += [dict(tags, id=p, y=int(a), p=float(b), base=float(c)) for a, b, c in zip(y.numpy(), pr, base)]
    return rows


feats, _ = R_COND['RANK']
S9 = {k: r_seqs(RD[k], feats) for k in RD}
t0 = time.time()
for t in RD:
    for cname, studies in [('mt3', list(RD)), ('mt2', [k for k in RD if k != 'SF'])]:
        if t == 'SF' and cname == 'mt2': continue
        name = f"R9_{t}__{cname}"
        if done(name): continue
        head_idx = {k: i for i, k in enumerate(studies)}
        ids = np.array(sorted(S9[t])); rows = []
        for r in range(R_REPEATS):
            perm = np.random.default_rng(SEED + r).permutation(ids)          # same folds as Part 7
            for k_, te in enumerate(np.array_split(perm, R_FOLDS)):
                tr = [p for p in ids if p not in set(te)]
                seqs = {(t, p): S9[t][p] for p in tr}
                for k in studies:
                    if k != t: seqs.update({(k, p): v for p, v in S9[k].items()})
                keys = list(seqs); head_of = {key: head_idx[key[0]] for key in keys}
                mdl = mt_train(seqs, keys, head_of, len(feats), len(studies), seed=SEED + 100 * r + k_)
                rows += mt_predict(mdl, S9[t], list(te), head_idx[t], repeat=r, fold=k_)
        r_save(name, rows); print(f"  {name:20s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 9B done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 22 — PART 9C: STACKING (logistic combiner over logistic-regression predictions)
# For each target fold k, the combiner is fitted on the other folds' out-of-fold predictions and applied to fold k.
# Base models: target-only (A_core), transfer from each other study, pooled (B_pool3, E_pool2), partial pooling (aug3, aug2).
# ════════════════════════════════════════════════════════
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))


for t in DATA:
    name = f"P9_{t}__stack__LR"
    if done(name): continue
    base = {'target': f"E_{t}_to_{t}__core__within__LR"}
    for s in DATA:
        if s != t: base[f'from_{s}'] = f"E_{s}_to_{t}__core__per_dataset__LR"
    base.update({'pool3': f"P8_{t}__B_pool3__LR", 'pool2': f"P8_{t}__E_pool2__LR",
                 'aug3': f"P9_{t}__aug3__LR", 'aug2': f"P9_{t}__aug2__LR"})
    base = {k: v for k, v in base.items() if done(v)}
    ref = load_pred(base['target']); ref = ref.assign(pos=ref.groupby('id').cumcount())
    J = ref[['id', 'pos', 'fold', 'y', 'base']].copy()
    for k, v in base.items():
        d = load_pred(v); d = d.assign(pos=d.groupby('id').cumcount())
        J = J.merge(d[['id', 'pos', 'p']].rename(columns={'p': k}), on=['id', 'pos'])
    cols = list(base); rows = []
    for k_ in sorted(J['fold'].unique()):
        trn, tst = J[J.fold != k_], J[J.fold == k_]
        comb = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced').fit(logit(trn[cols].to_numpy()), trn['y'])
        p = comb.predict_proba(logit(tst[cols].to_numpy()))[:, 1]
        rows.append(tst[['id', 'y', 'base', 'fold']].assign(p=p))
    out = pd.concat(rows); tmp = pred_path(name) + '.tmp'; out.to_csv(tmp, index=False); os.replace(tmp, pred_path(name))
    print(f"  {name:28s} base models: {cols}")
print("✅ Part 9C done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 23 — PART 9 RESULTS
# ════════════════════════════════════════════════════════
def pp_auc(d, col='p'):
    by = ['id'] + (['repeat'] if 'repeat' in d else [])
    a = d.groupby(by).apply(lambda g: auc(g['y'], g[col])).dropna()
    return a.groupby(level=0).mean()


def pooled_mean(d, col='p'):
    if 'repeat' not in d: return auc(d['y'], d[col])
    return np.nanmean([auc(g['y'], g[col]) for _, g in d.groupby('repeat')])


def fold_aucs(d, col='p'):
    return pd.Series({k: auc(g['y'], g[col]) for k, g in d.groupby('fold')})


def t_ci(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return (x.mean() - h, x.mean() + h)


rows = []
for t in DATA:
    A = load_pred(f"E_{t}_to_{t}__core__within__LR"); fa_A = fold_aucs(A); pp_A = per_person(A)
    rows.append(dict(target=t, model='A_core (target only)', fa_auc=round(fa_A.mean(), 3), diff='', CI='',
                     pp_median=round(pp_A.median(), 3), pp_diff='', share_better='', p=''))
    for lab, nm in [('B_pool3 (Part 8)', f"P8_{t}__B_pool3__LR"), ('E_pool2 (Part 8)', f"P8_{t}__E_pool2__LR"),
                    ('9A aug3', f"P9_{t}__aug3__LR"), ('9A aug2', f"P9_{t}__aug2__LR"), ('9C stack', f"P9_{t}__stack__LR")]:
        if not done(nm): continue
        Bd = load_pred(nm); fa_B = fold_aucs(Bd); dk = (fa_B - fa_A).dropna(); lo, hi = t_ci(dk)
        pp_B = per_person(Bd); cm = pp_B.index.intersection(pp_A.index); dd = (pp_B[cm] - pp_A[cm]).to_numpy()
        rows.append(dict(target=t, model=lab, fa_auc=round(fa_B.mean(), 3), diff=round(dk.mean(), 3),
                         CI=f"[{lo:+.3f}, {hi:+.3f}]", pp_median=round(pp_B.median(), 3),
                         pp_diff=round(float(np.median(dd)), 3), share_better=round(float((dd > 0).mean()), 2),
                         p=round(stats.wilcoxon(dd).pvalue, 4) if np.any(dd != 0) else 1.0))
P9 = pd.DataFrame(rows)
P9.to_csv(os.path.join(SAVE_DIR, 'part9_partial_pooling_LR.csv'), index=False)
pd.set_option('display.width', 250)
print("=== Part 9 — logistic models vs target-only (pooled AUC fold-matched; pp = per-participant median) ===")
print(P9.to_string(index=False))

rows = []
for t in RD:
    W = load_pred(f"R7_{t}_to_{t}__RANK"); aw = pp_auc(W)
    rows.append(dict(target=t, model='within (Part 7)', pp_median=round(aw.median(), 3), pooled=round(pooled_mean(W), 3),
                     pp_diff='', share_better='', p=''))
    for lab, nm in [('pool3 (Part 8)', f"R8_{t}__pool3"), ('pool2 (Part 8)', f"R8_{t}__pool2"),
                    ('9B multi-task mt3', f"R9_{t}__mt3"), ('9B multi-task mt2', f"R9_{t}__mt2")]:
        if not done(nm): continue
        Pd = load_pred(nm); ap = pp_auc(Pd); cm = ap.index.intersection(aw.index); dd = (ap[cm] - aw[cm]).to_numpy()
        rows.append(dict(target=t, model=lab, pp_median=round(ap[cm].median(), 3), pooled=round(pooled_mean(Pd), 3),
                         pp_diff=round(float(np.median(dd)), 3), share_better=round(float((dd > 0).mean()), 2),
                         p=round(stats.wilcoxon(dd).pvalue, 4) if np.any(dd != 0) else 1.0))
R9 = pd.DataFrame(rows)
R9.to_csv(os.path.join(SAVE_DIR, 'part9_multitask_gru.csv'), index=False)
print("\n=== Part 9 — ranking GRU: multi-task vs within-study (per participant, paired) ===")
print(R9.to_string(index=False))


---
## Part 9D — Does partial pooling help when the target study is small? (added 18 Oct 2026)

Theory predicts that partial pooling helps most when the target has little data of its own. We test this **as a separate,
pre-specified small-sample test**: for each target fold, train on only *n* randomly drawn target participants, then score
the held-out participants.

- Targets: EM (n = 3, 5, 10, 15) and AL (n = 3, 5, 10, 20, 40). 5 folds × 3 repeats; the same *n* participants for every model.
- Models: `target_only` (LR on the n target participants), `pool2` (naive pooling with the other prompted study),
  `aug2` (partial pooling with the other prompted study, **primary**), `aug3` (partial pooling with all three studies),
  and `source_only` (the other prompted study only, no target data) as a floor.
- Partial-pooling scale fixed in advance at 1.0 (no tuning: with n = 3–5 there is nothing to tune on).
- **Primary outcome:** per-participant AUC, `aug2` vs `target_only`, paired Wilcoxon at each n. Pooled AUC reported alongside.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 24 — PART 9D: SMALL-TARGET LEARNING CURVES FOR PARTIAL POOLING
# Needs Cells 1–4, 18 (build) and 20 (augment). Logistic regression only, CPU ≈ 20–40 min.
# ════════════════════════════════════════════════════════
SMALL_N = {'EM': [3, 5, 10, 15], 'AL': [3, 5, 10, 20, 40]}
SMALL_FOLDS, SMALL_REPEATS, SMALL_SCALE = 5, 3, 1.0
if QUICK_TEST: SMALL_N = {'EM': [3, 5], 'AL': [3, 5]}; SMALL_REPEATS = 1


def fit_lr(seqs):
    X, y = flat_next(seqs)
    return LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced', random_state=42).fit(X, y.astype(int))


def save_rows(name, rows):
    tmp = pred_path(name) + '.tmp'; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))


def has_both(seqs):
    y = np.concatenate([s[1][1:] for s in seqs]); return 0 < y.sum() < len(y)


t0 = time.time()
for tgt in ['EM', 'AL']:
    other = 'AL' if tgt == 'EM' else 'EM'
    full = {k: build(k, None, 'core')[0] for k in DATA}                     # other studies: all participants, own scaling
    ids = np.array([s_[2] for s_ in seqs_of(DATA[tgt], CORE)])
    # source-only floor (does not depend on n)
    name0 = f"P9D_{tgt}__source_only"
    if not done(name0):
        clf = fit_lr(full[other]); te_s, _ = build(tgt, list(ids), 'core')
        save_rows(name0, to_rows(predict_flat(clf, te_s)))
    for n in SMALL_N[tgt]:
        name = f"P9D_{tgt}__n{n:03d}"
        if done(name): continue
        rows = []
        for r in range(SMALL_REPEATS):
            for k_, (tr, te) in enumerate(KFold(SMALL_FOLDS, shuffle=True, random_state=SEED + r).split(ids)):
                tr_ids, te_ids = ids[tr], list(ids[te])
                rng = np.random.default_rng(SEED + 100 * r + k_)
                for _ in range(50):                                          # draw n participants with ≥1 lapse and ≥1 non-lapse
                    pick = list(rng.choice(tr_ids, min(n, len(tr_ids)), replace=False))
                    fit_s, musd = build(tgt, pick, 'core')
                    if has_both(fit_s): break
                else:
                    continue
                te_s, _ = build(tgt, te_ids, 'core', musd)
                tags = dict(n=n, repeat=r, fold=k_)
                rows += to_rows(predict_flat(fit_lr(fit_s), te_s), model='target_only', **tags)
                rows += to_rows(predict_flat(fit_lr(fit_s + full[other]), te_s), model='pool2', **tags)
                for mname, studies in [('aug2', [other]), ('aug3', [k for k in DATA if k != tgt])]:
                    clf = fit_aug({tgt: fit_s, **{k: full[k] for k in studies}}, SMALL_SCALE)
                    rows += to_rows(predict_flat(clf, augment(te_s, tgt, SMALL_SCALE)), model=mname, **tags)
        save_rows(name, rows); print(f"  {name:22s} {(time.time()-t0)/60:5.1f} min")
print("✅ Part 9D runs done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 25 — PART 9D RESULTS
# ════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

def pp_by_repeat(d):
    """Per-participant AUC, averaged over repeats."""
    a = d.groupby(['id', 'repeat']).apply(lambda g: auc(g['y'], g['p'])).dropna()
    return a.groupby(level=0).mean()


rows = []
for tgt in ['EM', 'AL']:
    S0 = load_pred(f"P9D_{tgt}__source_only"); pp_src = per_person(S0)
    for n in SMALL_N[tgt]:
        nm = f"P9D_{tgt}__n{n:03d}"
        if not done(nm): continue
        D = load_pred(nm); T = D[D.model == 'target_only']; ppT = pp_by_repeat(T)
        pooledT = np.nanmean([auc(g['y'], g['p']) for _, g in T.groupby(['repeat', 'fold'])])
        rows.append(dict(target=tgt, n=n, model='target_only', pp_median=round(ppT.median(), 3),
                         fold_auc=round(pooledT, 3), pp_diff='', share_better='', p=''))
        for m in ['pool2', 'aug2', 'aug3']:
            M_ = D[D.model == m]; ppM = pp_by_repeat(M_); cm = ppM.index.intersection(ppT.index)
            dd = (ppM[cm] - ppT[cm]).to_numpy()
            pooledM = np.nanmean([auc(g['y'], g['p']) for _, g in M_.groupby(['repeat', 'fold'])])
            rows.append(dict(target=tgt, n=n, model=m, pp_median=round(ppM[cm].median(), 3), fold_auc=round(pooledM, 3),
                             pp_diff=round(float(np.median(dd)), 3), share_better=round(float((dd > 0).mean()), 2),
                             p=round(stats.wilcoxon(dd).pvalue, 4) if np.any(dd != 0) else 1.0))
        cm = pp_src.index.intersection(ppT.index)
        rows.append(dict(target=tgt, n=n, model='source_only (n=0)', pp_median=round(pp_src[cm].median(), 3),
                         fold_auc=round(auc(S0['y'], S0['p']), 3), pp_diff='', share_better='', p=''))
P9D = pd.DataFrame(rows)
P9D.to_csv(os.path.join(SAVE_DIR, 'part9d_small_target.csv'), index=False)
pd.set_option('display.width', 250)
print("=== Part 9D — small target samples: each model vs target-only on the same n participants ===")
print("    (pp = per-participant AUC, paired; fold_auc = mean pooled AUC over folds × repeats)")
print(P9D.to_string(index=False))

# learning-curve figure (per-participant median AUC vs n)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=False)
for ax, tgt in zip(axes, ['EM', 'AL']):
    sub = P9D[P9D.target == tgt]
    for m, mk in [('target_only', 'o'), ('pool2', 's'), ('aug2', '^'), ('aug3', 'v')]:
        s_ = sub[sub.model == m]; ax.plot(s_['n'], s_['pp_median'].astype(float), marker=mk, label=m)
    src = sub[sub.model == 'source_only (n=0)']['pp_median'].astype(float)
    if len(src): ax.axhline(src.iloc[0], ls='--', c='grey', lw=1, label='source only')
    ax.set_xscale('log'); ax.minorticks_off(); ax.set_xticks(SMALL_N[tgt]); ax.set_xticklabels(SMALL_N[tgt])
    ax.set_xlabel(f'{tgt} training participants (n)'); ax.set_ylabel('Per-participant AUC (median)'); ax.set_title(tgt)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=5, fontsize=8, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.savefig(os.path.join(SAVE_DIR, 'fig_part9d_small_target.png'), dpi=300); plt.show()


In [ ]:
# ════════════════════════════════════════════════════════
# TABLE 10 — UNCERTAINTY FOR THE SMALL-STUDY TRANSFER RESULTS
# Run in Transfer_Notebook after Cell 1 (it only needs SAVE_DIR). Reads the saved Part 9D predictions; no retraining.
# For each target study and n: AUC on each of the 15 fold-repeat test sets, then for naive and partial pooling the
# mean paired difference from target-only training with a 95% t-interval over the 15 fold-repeats.
# (Fold-repeats share participants, so the interval is approximate, like the fold-level intervals in the main benchmark.)
# ════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

SAVE_DIR = globals().get('SAVE_DIR', 'transfer_results')
PRED = os.path.join(SAVE_DIR, 'predictions')
SMALL_N = {'EM': [3, 5, 10, 15], 'AL': [3, 5, 10, 20, 40]}
NAME = {'EM': 'EMASENS', 'AL': 'Alcohol'}
MODELS = {'target_only': 'Target only', 'pool2': 'Naive pooling', 'aug2': 'Partial pooling'}


def fold_aucs(D, model):
    out = {}
    for (r, k), g in D[D.model == model].groupby(['repeat', 'fold']):
        if g.y.nunique() == 2: out[(r, k)] = roc_auc_score(g.y, g.p)
    return pd.Series(out)


rows = []
for tgt, ns in SMALL_N.items():
    for n in ns:
        f = os.path.join(PRED, f"P9D_{tgt}__n{n:03d}.csv")
        if not os.path.exists(f): print("missing", f); continue
        D = pd.read_csv(f, dtype={'id': str})
        A = {m: fold_aucs(D, m) for m in MODELS}
        row = {'Target': NAME[tgt], 'n': n}
        for m, lab in MODELS.items():
            row[lab] = f"{A[m].mean():.3f}"
        for m in ['pool2', 'aug2']:
            common = A[m].index.intersection(A['target_only'].index)
            d = (A[m][common] - A['target_only'][common]).to_numpy()
            half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))
            row[f"{MODELS[m]} − target (95% CI)"] = f"{d.mean():+.3f} ({d.mean() - half:+.3f} to {d.mean() + half:+.3f})"
        row['fold-repeats'] = len(common)
        rows.append(row)

T10 = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 40)
print(T10.to_string(index=False))
T10.to_csv(os.path.join(SAVE_DIR, 'table10_with_ci.csv'), index=False)
print('Saved table10_with_ci.csv')

---
## Part 10 — Personalisation: revisiting Perski et al.'s Objectives 2–4 (exploratory, added 18 Oct 2026)

Perski et al. (2023) used the Smoke Free data to test a group model on unseen people (Objective 2), individual models
(Objective 3, feasible for 39/791 people) and a hybrid "warm start" (Objective 4, group data + a random 20% of the person's
entries). Here the same questions are asked for **next-entry** prediction with a **realistic warm start**: the first *k*
entries of each held-out person are used to personalise the model, which is then tested on that person's later entries.

**Two starting points:** a model pretrained on Smoke Free only (`own`) and on all three studies (`all3`; other studies
standardised with their own statistics). Participants are split into 5 folds; pretraining never sees the test fold.

| Model | Meaning |
|---|---|
| `group` | pretrained model, no personalisation (Objective 2) |
| `person_rate` | the person's running lapse rate (simple personal baseline) |
| `individual` | LR trained only on the person's first *k* entries; only when they contain a lapse and a non-lapse (Objective 3) |
| `hybrid` | pretrained data + the person's first *k* entries, refitted with equal weight (Perski et al.'s Objective 4) |
| `tl_shrink_l1`, `tl_shrink_l10` | transfer learning: start from the pretrained coefficients and adapt them to the person's first *k* entries, pulled back toward the pretrained values (λ = 1 weak, λ = 10 strong). Works for everyone, even with no lapse yet |
| `finetune` (GRU) | the pretrained ranking GRU fine-tuned on the person's first *k* entries (as in Part 7B) |

Evaluated people: those with a lapse and a non-lapse among the entries after the first *k*. Outcome: per-participant AUC,
paired against `group` from the same starting point; `all3` vs `own` compared for each model.
Run Cells 1–4, 16, 18, then 26, 27, 28.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 26 — PART 10A: PERSONALISATION WITH LOGISTIC REGRESSION (two starting points: own study vs all three)
# ════════════════════════════════════════════════════════
import warnings
from copy import deepcopy
from scipy.optimize import minimize
from sklearn.exceptions import ConvergenceWarning

PERS_TARGETS = ['SF']            # add 'EM', 'AL' to repeat on the prompted studies
PERS_K = [5, 10, 20]             # first k entries of each test person are used for personalisation
PERS_FOLDS = 5
PERS_LAMBDAS = [1.0, 10.0]       # pull toward the pretrained coefficients (larger = stay closer)
if QUICK_TEST: PERS_K = [5, 10]


def save_rows(name, rows):
    tmp = pred_path(name) + '.tmp'; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))


def shrink_fit(X, y, w0, b0, lam):
    """Person-level transfer: minimise the person's log-loss + lam/2 * ||(w, b) - (w0, b0)||^2."""
    d = X.shape[1]
    def f(v):
        w, b = w0 + v[:d], b0 + v[d]; z = X @ w + b; p = 1 / (1 + np.exp(-z))
        loss = np.sum(np.logaddexp(0, z) - y * z) + 0.5 * lam * np.sum(v ** 2)
        g = p - y
        return loss, np.r_[X.T @ g + lam * v[:d], g.sum() + lam * v[d]]
    v = minimize(f, np.zeros(d + 1), jac=True, method='L-BFGS-B').x
    return w0 + v[:d], b0 + v[d]


def lin_pred(X, w, b): return 1 / (1 + np.exp(-(X @ w + b)))


t0 = time.time()
for t in PERS_TARGETS:
    all_ids = [s_[2] for s_ in seqs_of(DATA[t], CORE)]
    folds = list(KFold(PERS_FOLDS, shuffle=True, random_state=SEED).split(all_ids))
    others = [s_ for k in DATA if k != t for s_ in build(k, None, 'core')[0]]
    for start in ['own', 'all3']:
        name = f"P10_{t}__{start}"
        if done(name): continue
        rows = []
        for f_, (tr, te) in enumerate(folds):
            tr_ids = [all_ids[i] for i in tr]; te_ids = [all_ids[i] for i in te]
            tr_s, musd = build(t, tr_ids, 'core'); te_s, _ = build(t, te_ids, 'core', musd)
            Xp, yp = flat_next(tr_s + (others if start == 'all3' else [])); yp = yp.astype(int)
            g = LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced', random_state=42).fit(Xp, yp)
            w0, b0 = g.coef_[0].astype(float), float(g.intercept_[0])
            for X, y, pid, rr in te_s:
                Xq, yq, bq = X[:-1], y[1:].astype(float), rr[:-1]          # pair i: entry i -> lapse at entry i+1
                for k in PERS_K:
                    a = k - 1                                               # pairs that lie entirely within the first k entries
                    if len(yq) <= a + 1: continue
                    Xa, ya, Xt, yt, bt = Xq[:a], yq[:a], Xq[a:], yq[a:], bq[a:]
                    if not (0 < yt.sum() < len(yt)): continue               # AUC needs both outcomes after the first k entries
                    tag = dict(id=pid, fold=f_, k=k)
                    def add(model, p):
                        rows.extend(dict(tag, model=model, y=int(u), p=float(v_), base=float(c)) for u, v_, c in zip(yt, p, bt))
                    add('group', lin_pred(Xt, w0, b0))
                    add('person_rate', bt)
                    if 0 < ya.sum() < len(ya):
                        ind = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced').fit(Xa, ya.astype(int))
                        add('individual', ind.predict_proba(Xt)[:, 1])
                    hy = deepcopy(g).set_params(warm_start=True, max_iter=300)
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore', ConvergenceWarning)
                        hy.fit(np.vstack([Xp, Xa]), np.r_[yp, ya.astype(int)])
                    add('hybrid', hy.predict_proba(Xt)[:, 1])
                    for lam in PERS_LAMBDAS:
                        w, b = shrink_fit(Xa.astype(float), ya, w0, b0, lam)
                        add(f'tl_shrink_l{lam:g}', lin_pred(Xt, w, b))
            print(f"  {name}: fold {f_ + 1}/{PERS_FOLDS} done  {(time.time()-t0)/60:5.1f} min")
        save_rows(name, rows)
print("✅ Part 10A done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 27 — PART 10B: PERSONALISATION OF THE RANKING GRU (fine-tune the pretrained model on the first k entries)
# Needs Cell 16 (RD, r_seqs, r_train, R_* settings, r_save). Eligible people as in Part 7.
# ════════════════════════════════════════════════════════
feats, _ = R_COND['RANK']
S10 = {k: r_seqs(RD[k], feats) for k in RD}
t0 = time.time()


def gru_probs(model, x):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model.classify(x.unsqueeze(0).to(device), [len(x)]))[0].cpu().numpy()


for t in PERS_TARGETS:
    ids = np.array(sorted(S10[t]))
    for start in ['own', 'all3']:
        name = f"R10_{t}__{start}"
        if done(name): continue
        rows = []
        for f_, te in enumerate(np.array_split(np.random.default_rng(SEED).permutation(ids), R_FOLDS)):
            te_set = set(te); seqs = {(t, p): S10[t][p] for p in ids if p not in te_set}
            if start == 'all3':
                for k in RD:
                    if k != t: seqs.update({(k, p): v for p, v in S10[k].items()})
            pre = r_train(seqs, list(seqs), len(feats), True, seed=SEED + f_)
            init = deepcopy(pre.state_dict())
            for p in te:
                x, y, base = S10[t][p]; yn = y.numpy(); pg = gru_probs(pre, x)
                for k in PERS_K:
                    a = k - 1
                    if len(yn) <= a + 1: continue
                    yt = yn[a:]
                    if not (0 < yt.sum() < len(yt)): continue
                    tag = dict(id=p, fold=f_, k=k)
                    def add(model, pr):
                        rows.extend(dict(tag, model=model, y=int(u), p=float(v_), base=float(c))
                                    for u, v_, c in zip(yt, pr[a:], base[a:]))
                    add('group', pg); add('person_rate', base)
                    ya = y[:a]; both = bool(0 < ya.sum() < len(ya))
                    ft = r_train({p: (x[:a], ya, base[:a])}, [p], len(feats), both, seed=SEED + f_, init=init,
                                 ssl=False, epochs=R_FT_EPOCHS, lr=R_FT_LR)
                    add('finetune', gru_probs(ft, x))
            print(f"  {name}: fold {f_ + 1}/{R_FOLDS} done  {(time.time()-t0)/60:5.1f} min")
        r_save(name, rows)
print("✅ Part 10B done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 28 — PART 10 RESULTS
# ════════════════════════════════════════════════════════
def pp(d):
    return pd.Series({pid: auc(g['y'], g['p']) for pid, g in d.groupby('id')}).dropna()


def paired(a, b):
    cm = a.index.intersection(b.index); dd = (a[cm] - b[cm]).to_numpy()
    if len(dd) == 0: return np.nan, np.nan, np.nan, 0
    p = stats.wilcoxon(dd).pvalue if np.any(dd != 0) else 1.0
    return float(np.median(dd)), float((dd > 0).mean()), p, len(dd)


rows = []
for t in PERS_TARGETS:
    for method, prefix in [('LR', 'P10'), ('GRU', 'R10')]:
        D = {s: load_pred(f"{prefix}_{t}__{s}") for s in ['own', 'all3'] if done(f"{prefix}_{t}__{s}")}
        for start, d in D.items():
            for k in sorted(d['k'].unique()):
                dk = d[d.k == k]; G = pp(dk[dk.model == 'group'])
                for m in dk['model'].unique():
                    M_ = pp(dk[dk.model == m]); md, sb, p, n = paired(M_, G)
                    fold_auc = np.nanmean([auc(g['y'], g['p']) for _, g in dk[dk.model == m].groupby('fold')])
                    row = dict(target=t, method=method, start=start, k=k, model=m, n_people=len(M_),
                               pp_median=round(M_.median(), 3), fold_auc=round(fold_auc, 3))
                    if m != 'group':
                        row.update(vs_group=round(md, 3), share_better=round(sb, 2), p=round(p, 4))
                    if start == 'all3' and 'own' in D:                 # same model, other starting point
                        do = D['own']; Mo = pp(do[(do.k == k) & (do.model == m)])
                        md2, sb2, p2, _ = paired(M_, Mo)
                        row.update(all3_vs_own=round(md2, 3), p_all3_vs_own=round(p2, 4))
                    rows.append(row)
P10 = pd.DataFrame(rows).fillna('')
order = {m: i for i, m in enumerate(['group', 'person_rate', 'individual', 'hybrid', 'tl_shrink_l1', 'tl_shrink_l10', 'finetune'])}
P10 = P10.sort_values(['target', 'method', 'k', 'start', 'model'], key=lambda s: s.map(order) if s.name == 'model' else s)
P10.to_csv(os.path.join(SAVE_DIR, 'part10_personalisation.csv'), index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 300)
print("=== Part 10 — personalisation on each person's first k entries; tested on later entries ===")
print("    pp = per-participant AUC; vs_group = median paired difference vs the same start's group model;")
print("    all3_vs_own = same model, pretrained on all three studies minus pretrained on own study only")
print(P10.to_string(index=False))


---
## Part 10C — Why does person-level shrinkage work? A personal-baseline check (exploratory, added 18 Oct 2026)

When a person's first *k* entries are mostly non-lapses, the shrinkage update in Cell 26 changes the coefficients by roughly
**minus the average of that person's early entries**. A later entry then gets a lower score the more it resembles the
person's usual entries, and a higher score the more it departs from them. If that is the whole story, a model that simply adds
"how similar is this entry to the person's own early average?" to the group model should match the shrinkage result.

| Model | Score for a later entry *x* of a person with early-entry average *m* |
|---|---|
| `baseline_sim` | −(*m* · *x*): similarity to the person's early average, sign flipped (no learning) |
| `baseline_dist` | ‖*x* − *m*‖: distance from the person's early average (no learning) |
| `group_plus_sim` | group logit + β · (−*m* · *x*), with β learned on the training participants |
| `group_plus_dist` | group logit + β · ‖*x* − *m*‖, with β learned on the training participants |

Same folds, people and *k* as Cell 26, both starting points. Compared with `group` and `tl_shrink_l1` in Cell 30.
**Reading it:** if `group_plus_sim` ≈ `tl_shrink_l1`, the personal-baseline explanation holds and the simpler model can be reported.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 29 — PART 10C: PERSONAL-BASELINE MECHANISM CHECK (needs Cell 26 run in this session)
# ════════════════════════════════════════════════════════
def baseline_feats(Xa, Xt):
    m = Xa.mean(0)
    return -(Xt @ m), np.linalg.norm(Xt - m, axis=1)


t0 = time.time()
for t in PERS_TARGETS:
    all_ids = [s_[2] for s_ in seqs_of(DATA[t], CORE)]
    folds = list(KFold(PERS_FOLDS, shuffle=True, random_state=SEED).split(all_ids))    # same folds as Cell 26
    others = [s_ for k in DATA if k != t for s_ in build(k, None, 'core')[0]]
    for start in ['own', 'all3']:
        name = f"P10M_{t}__{start}"
        if done(name): continue
        rows = []
        for f_, (tr, te) in enumerate(folds):
            tr_ids = [all_ids[i] for i in tr]; te_ids = [all_ids[i] for i in te]
            tr_s, musd = build(t, tr_ids, 'core'); te_s, _ = build(t, te_ids, 'core', musd)
            Xp, yp = flat_next(tr_s + (others if start == 'all3' else []))
            g = LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced', random_state=42).fit(Xp, yp.astype(int))
            w0, b0 = g.coef_[0].astype(float), float(g.intercept_[0])
            for k in PERS_K:
                a = k - 1
                # learn beta on the training participants of the target study (their own first k entries as baseline)
                Fb, Yb = [], []
                for X, y, pid, rr in tr_s:
                    Xq, yq = X[:-1], y[1:]
                    if len(yq) <= a + 1: continue
                    sim, dist = baseline_feats(Xq[:a], Xq[a:])
                    Fb.append(np.c_[Xq[a:] @ w0 + b0, sim, dist]); Yb.append(yq[a:])
                Fb, Yb = np.vstack(Fb), np.concatenate(Yb).astype(int)
                c_sim = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Fb[:, [0, 1]], Yb)
                c_dist = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Fb[:, [0, 2]], Yb)
                for X, y, pid, rr in te_s:
                    Xq, yq, bq = X[:-1], y[1:].astype(float), rr[:-1]
                    if len(yq) <= a + 1: continue
                    Xa, Xt, yt, bt = Xq[:a], Xq[a:], yq[a:], bq[a:]
                    if not (0 < yt.sum() < len(yt)): continue
                    sim, dist = baseline_feats(Xa, Xt); lg = Xt @ w0 + b0
                    tag = dict(id=pid, fold=f_, k=k)
                    for model, p in [('baseline_sim', sim), ('baseline_dist', dist),
                                     ('group_plus_sim', c_sim.decision_function(np.c_[lg, sim])),
                                     ('group_plus_dist', c_dist.decision_function(np.c_[lg, dist]))]:
                        rows.extend(dict(tag, model=model, y=int(u), p=float(v_), base=float(c)) for u, v_, c in zip(yt, p, bt))
                if f_ == 0:
                    print(f"  {name} k={k}: beta_sim {c_sim.coef_[0][1]:+.3f}  beta_dist {c_dist.coef_[0][1]:+.3f}"
                          f"  (positive = less like the person's early entries -> higher risk)")
            print(f"  {name}: fold {f_ + 1}/{PERS_FOLDS} done  {(time.time()-t0)/60:5.1f} min")
        save_rows(name, rows)
print("✅ Part 10C done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 30 — PART 10C RESULTS: mechanism models vs group and vs shrinkage (same people)
# ════════════════════════════════════════════════════════
rows = []
for t in PERS_TARGETS:
    for start in ['own', 'all3']:
        if not (done(f"P10_{t}__{start}") and done(f"P10M_{t}__{start}")): continue
        A = pd.concat([load_pred(f"P10_{t}__{start}"), load_pred(f"P10M_{t}__{start}")])
        for k in sorted(A['k'].unique()):
            Ak = A[A.k == k]; G = pp(Ak[Ak.model == 'group']); S = pp(Ak[Ak.model == 'tl_shrink_l1'])
            for m in ['group', 'tl_shrink_l1', 'baseline_sim', 'baseline_dist', 'group_plus_sim', 'group_plus_dist']:
                M_ = pp(Ak[Ak.model == m]); row = dict(target=t, start=start, k=k, model=m, n_people=len(M_),
                                                      pp_median=round(M_.median(), 3))
                if m != 'group':
                    md, sb, p, _ = paired(M_, G); row.update(vs_group=round(md, 3), share_better_g=round(sb, 2), p_g=round(p, 4))
                if m not in ('group', 'tl_shrink_l1'):
                    md, sb, p, _ = paired(M_, S); row.update(vs_shrink=round(md, 3), p_s=round(p, 4))
                    cm = M_.index.intersection(S.index)
                    row.update(corr_with_shrink=round(float(np.corrcoef(M_[cm], S[cm])[0, 1]), 2) if len(cm) > 2 else '')
                rows.append(row)
P10M = pd.DataFrame(rows).fillna('')
P10M.to_csv(os.path.join(SAVE_DIR, 'part10c_mechanism.csv'), index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 300)
print("=== Part 10C — personal-baseline models (pp = per-participant AUC median; paired medians) ===")
print("    vs_group / vs_shrink = median paired difference; corr_with_shrink = correlation of per-person AUCs with tl_shrink_l1")
print(P10M.to_string(index=False))


---
## Part 11 — Same-entry classification (Perski et al.) vs next-entry prediction, personalisation and history check (exploratory)

Perski et al. **classified** the lapse at the same entry (features and lapse reported together); our work **predicts** the
lapse at the next entry. Part 11 repeats the Part 10 comparison under both tasks, on all three studies and both starting
points, and adds the history check for the personal-baseline effect:

- `classify`: entry *t* features → lapse at entry *t* (lapse-history features use entries before *t* only; the person's
  running lapse rate is taken *before* entry *t*).
- `predict`: entry *t* features → lapse at entry *t*+1 (as in Parts 1–10).
- Personal-baseline scores computed **with** all features (`baseline_sim`, `baseline_dist`) and **without the four
  lapse-history features** (`*_nohist`). If the Smoke Free gain disappears without history, the "personal baseline" effect
  is really lapse persistence (runs of lapses).

Same folds, *k* values and λ values as Part 10. Run Cells 1–4, 16, 18, then 31, 32, 33.


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 31 — PART 11A: LOGISTIC MODELS FOR BOTH TASKS (classify = same entry, predict = next entry)
# ════════════════════════════════════════════════════════
import warnings
from copy import deepcopy
from scipy.optimize import minimize
from sklearn.exceptions import ConvergenceWarning

TASKS = ['classify', 'predict']
P11_TARGETS = ['SF', 'EM', 'AL']
P11_K, P11_FOLDS, P11_LAMBDAS = [5, 10, 20], 5, [1.0, 10.0]
HIST = ['prior_lapse', 'lapse_lag2', 'lapse_lag3', 'days_since_last_lapse']
NOHIST_IDX = [i for i, f in enumerate(CORE) if f not in HIST]
if QUICK_TEST: P11_K = [5, 10]


def save_rows(name, rows):
    tmp = pred_path(name) + '.tmp'; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, pred_path(name))


def task_rows(X, y, rr, task):
    """Rows (features, outcome, person's running lapse rate known at that time) and the number of rows in the first k entries."""
    if task == 'predict':
        return X[:-1], y[1:].astype(float), rr[:-1], (lambda k: k - 1)
    return X, y.astype(float), np.r_[0.0, rr[:-1]], (lambda k: k)


def task_flat(seqs, task):
    R = [task_rows(X, y, rr, task) for X, y, pid, rr in seqs]
    return np.vstack([r[0] for r in R]), np.concatenate([r[1] for r in R]).astype(int)


def shrink_fit(X, y, w0, b0, lam):
    d = X.shape[1]
    def f(v):
        w, b = w0 + v[:d], b0 + v[d]; z = X @ w + b; p = 1 / (1 + np.exp(-z))
        g = p - y
        return np.sum(np.logaddexp(0, z) - y * z) + 0.5 * lam * np.sum(v ** 2), np.r_[X.T @ g + lam * v[:d], g.sum() + lam * v[d]]
    v = minimize(f, np.zeros(d + 1), jac=True, method='L-BFGS-B').x
    return w0 + v[:d], b0 + v[d]


def lin_pred(X, w, b): return 1 / (1 + np.exp(-(X @ w + b)))


def base_scores(Xa, Xt, idx=None):
    if idx is not None: Xa, Xt = Xa[:, idx], Xt[:, idx]
    m = Xa.mean(0); return -(Xt @ m), np.linalg.norm(Xt - m, axis=1)


t0 = time.time()
for task in TASKS:
    for t in P11_TARGETS:
        all_ids = [s_[2] for s_ in seqs_of(DATA[t], CORE)]
        folds = list(KFold(P11_FOLDS, shuffle=True, random_state=SEED).split(all_ids))
        others = [s_ for k in DATA if k != t for s_ in build(k, None, 'core')[0]]
        for start in ['own', 'all3']:
            name = f"P11_{task}_{t}__{start}"
            if done(name): continue
            rows = []
            for f_, (tr, te) in enumerate(folds):
                tr_ids = [all_ids[i] for i in tr]; te_ids = [all_ids[i] for i in te]
                tr_s, musd = build(t, tr_ids, 'core'); te_s, _ = build(t, te_ids, 'core', musd)
                Xp, yp = task_flat(tr_s + (others if start == 'all3' else []), task)
                g = LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced', random_state=42).fit(Xp, yp)
                w0, b0 = g.coef_[0].astype(float), float(g.intercept_[0])
                for X, y, pid, rr in te_s:
                    Xq, yq, bq, cut = task_rows(X, y, rr, task)
                    for k in P11_K:
                        a = cut(k)
                        if len(yq) <= a + 1: continue
                        Xa, ya, Xt, yt, bt = Xq[:a], yq[:a], Xq[a:], yq[a:], bq[a:]
                        if not (0 < yt.sum() < len(yt)): continue
                        tag = dict(id=pid, fold=f_, k=k)
                        def add(model, p):
                            rows.extend(dict(tag, model=model, y=int(u), p=float(v_), base=float(c)) for u, v_, c in zip(yt, p, bt))
                        add('group', lin_pred(Xt, w0, b0)); add('person_rate', bt)
                        if 0 < ya.sum() < len(ya):
                            add('individual', LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
                                .fit(Xa, ya.astype(int)).predict_proba(Xt)[:, 1])
                        hy = deepcopy(g).set_params(warm_start=True, max_iter=300)
                        with warnings.catch_warnings():
                            warnings.simplefilter('ignore', ConvergenceWarning)
                            hy.fit(np.vstack([Xp, Xa]), np.r_[yp, ya.astype(int)])
                        add('hybrid', hy.predict_proba(Xt)[:, 1])
                        for lam in P11_LAMBDAS:
                            w, b = shrink_fit(Xa.astype(float), ya, w0, b0, lam); add(f'tl_shrink_l{lam:g}', lin_pred(Xt, w, b))
                        s1, d1 = base_scores(Xa, Xt); s2, d2 = base_scores(Xa, Xt, NOHIST_IDX)
                        add('baseline_sim', s1); add('baseline_dist', d1)
                        add('baseline_sim_nohist', s2); add('baseline_dist_nohist', d2)
            save_rows(name, rows); print(f"  {name:28s} {(time.time()-t0)/60:6.1f} min")
print("✅ Part 11A done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 32 — PART 11B: RANKING GRU FOR SAME-ENTRY CLASSIFICATION (next-entry GRU results come from Cell 27 / R10 files)
# Needs Cell 16. In the classification task the lapse-history inputs lapse_last5 and cum_rate are lagged by one entry,
# so the current lapse is never an input.
# ════════════════════════════════════════════════════════
import torch.nn.functional as F                              # re-import in case F was overwritten earlier

def c_seqs(d, feats):
    d = d.copy()
    for c in ['lapse_last5', 'cum_rate']:                    # minimum of the standardised column = raw value 0
        d[c] = d.groupby('id')[c].shift(1).fillna(d[c].min())
    return {pid: (torch.tensor(g[feats].to_numpy(np.float32)), torch.tensor(g['label'].to_numpy(np.float32)),
                  g['cum_rate'].to_numpy()) for pid, g in d.groupby('id', sort=True) if len(g) >= 2}


def gru_probs(model, x):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model.classify(x.unsqueeze(0).to(device), [len(x)]))[0].cpu().numpy()


feats, _ = R_COND['RANK']
S11 = {k: c_seqs(RD[k], feats) for k in RD}
t0 = time.time()
for t in P11_TARGETS:
    ids = np.array(sorted(S11[t]))
    for start in ['own', 'all3']:
        name = f"R11_classify_{t}__{start}"
        if done(name): continue
        rows = []
        for f_, te in enumerate(np.array_split(np.random.default_rng(SEED).permutation(ids), R_FOLDS)):
            te_set = set(te); seqs = {(t, p): S11[t][p] for p in ids if p not in te_set}
            if start == 'all3':
                for k in RD:
                    if k != t: seqs.update({(k, p): v for p, v in S11[k].items()})
            pre = r_train(seqs, list(seqs), len(feats), True, seed=SEED + f_); init = deepcopy(pre.state_dict())
            for p in te:
                x, y, base = S11[t][p]; yn = y.numpy(); pg = gru_probs(pre, x)
                for k in P11_K:
                    if len(yn) <= k + 1: continue
                    yt = yn[k:]
                    if not (0 < yt.sum() < len(yt)): continue
                    tag = dict(id=p, fold=f_, k=k)
                    def add(model, pr):
                        rows.extend(dict(tag, model=model, y=int(u), p=float(v_), base=float(c))
                                    for u, v_, c in zip(yt, pr[k:], base[k:]))
                    add('group', pg); add('person_rate', base)
                    ya = y[:k]; both = bool(0 < ya.sum() < len(ya))
                    ft = r_train({p: (x[:k], ya, base[:k])}, [p], len(feats), both, seed=SEED + f_, init=init,
                                 ssl=False, epochs=R_FT_EPOCHS, lr=R_FT_LR)
                    add('finetune', gru_probs(ft, x))
            print(f"  {name}: fold {f_ + 1}/{R_FOLDS}  {(time.time()-t0)/60:5.1f} min")
        r_save(name, rows)
print("✅ Part 11B done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 33 — PART 11 RESULTS: both tasks side by side
# ════════════════════════════════════════════════════════
def pp(d):
    return pd.Series({pid: auc(g['y'], g['p']) for pid, g in d.groupby('id')}).dropna()


def paired(a, b):
    cm = a.index.intersection(b.index); dd = (a[cm] - b[cm]).to_numpy()
    if len(dd) == 0: return np.nan, np.nan, np.nan
    return float(np.median(dd)), float((dd > 0).mean()), (stats.wilcoxon(dd).pvalue if np.any(dd != 0) else 1.0)


def gru_file(task, t, start):
    return f"R11_classify_{t}__{start}" if task == 'classify' else f"R10_{t}__{start}"


rows = []
for task in TASKS:
    for t in P11_TARGETS:
        for method in ['LR', 'GRU']:
            D = {}
            for s in ['own', 'all3']:
                nm = f"P11_{task}_{t}__{s}" if method == 'LR' else gru_file(task, t, s)
                if done(nm): D[s] = load_pred(nm)
            for start, d in D.items():
                for k in sorted(d['k'].unique()):
                    dk = d[d.k == k]; G = pp(dk[dk.model == 'group'])
                    for m in dk['model'].unique():
                        M_ = pp(dk[dk.model == m]); md, sb, p = paired(M_, G)
                        row = dict(task=task, target=t, method=method, start=start, k=k, model=m, n_people=len(M_),
                                   pp_median=round(M_.median(), 3),
                                   fold_auc=round(np.nanmean([auc(g_['y'], g_['p']) for _, g_ in dk[dk.model == m].groupby('fold')]), 3))
                        if m != 'group': row.update(vs_group=round(md, 3), share_better=round(sb, 2), p=round(p, 4))
                        if start == 'all3' and 'own' in D:
                            do = D['own']; md2, _, p2 = paired(M_, pp(do[(do.k == k) & (do.model == m)]))
                            row.update(all3_vs_own=round(md2, 3), p_all3_vs_own=round(p2, 4))
                        rows.append(row)
P11 = pd.DataFrame(rows)
P11.to_csv(os.path.join(SAVE_DIR, 'part11_classify_vs_predict.csv'), index=False)
print(f"Full table ({len(P11)} rows) saved to part11_classify_vs_predict.csv")

# compact view 1: per-participant median AUC, own-study start
own = P11[P11.start == 'own'].copy(); own['col'] = own['method'] + ':' + own['model']
cols = ['LR:group', 'LR:hybrid', 'LR:individual', 'LR:tl_shrink_l1', 'LR:tl_shrink_l10', 'LR:baseline_sim',
        'LR:baseline_sim_nohist', 'LR:baseline_dist', 'LR:baseline_dist_nohist', 'GRU:group', 'GRU:finetune']
pd.set_option('display.width', 300); pd.set_option('display.max_columns', 30)
print("\n=== Per-participant median AUC (own-study start) ===")
print(own.pivot_table(index=['task', 'target', 'k'], columns='col', values='pp_median').reindex(columns=cols).round(3).to_string())

# compact view 2: difference vs the group model of the same method, with p-value
own['d'] = own.apply(lambda r: '' if r['model'] == 'group' or pd.isna(r.get('vs_group')) else f"{r['vs_group']:+.3f} ({r['p']:.3f})", axis=1)
print("\n=== Median paired difference vs group model of the same method (p), own-study start ===")
print(own.pivot_table(index=['task', 'target', 'k'], columns='col', values='d', aggfunc='first')
      .reindex(columns=[c for c in cols if not c.endswith(':group')]).fillna('').to_string())

# compact view 3: does pretraining on all three studies help?
a3 = P11[(P11.start == 'all3') & P11.model.isin(['group', 'tl_shrink_l1', 'finetune'])].copy()
a3['d'] = a3.apply(lambda r: f"{r['all3_vs_own']:+.3f} ({r['p_all3_vs_own']:.3f})" if pd.notna(r.get('all3_vs_own')) else '', axis=1)
a3['col'] = a3['method'] + ':' + a3['model']
print("\n=== all3 minus own start, same model (p) ===")
print(a3.pivot_table(index=['task', 'target', 'k'], columns='col', values='d', aggfunc='first').fillna('').to_string())


## References

- Perski, O., Li, K., Pontikos, N., Simons, D., Goldstein, S. P., Naughton, F., & Brown, J. (2023). Classification
  of lapses in smokers attempting to stop: A supervised machine learning approach using data from a popular
  smoking cessation smartphone app. *Nicotine & Tobacco Research, 25*(7), 1330–1339.
- Perski, O., Kale, D., Leppin, C., Okpako, T., Simons, D., Goldstein, S. P., Hekler, E., & Brown, J. (2024).
  Supervised machine learning to predict smoking lapses from ecological momentary assessments and sensor data.
  *PLOS Digital Health, 3*(8), e0000594. Data: Zenodo 12532026 (CC BY 4.0).
- Wyant, K., Sant'Ana, S. J., Fronk, G. E., & Curtin, J. J. (2024). Machine learning models for temporally precise
  lapse prediction in alcohol use disorder. *Journal of Psychopathology and Clinical Science, 133*(7), 527–540.
  Data: https://osf.io/w5h9y.
